# functions

In [1]:
import pandas as pd
import glob
import os
# Set option to display all columns
pd.set_option('display.max_columns', None)


In [2]:
import pandas as pd
import numpy as np
from urllib.parse import urlparse
import re
import unicodedata

DOI_CORE_RE = re.compile(r"(10\.\d{4,9}/\S+)", re.IGNORECASE)

def _normalize_doi_raw(x: str) -> str | None:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip()
    s = unicodedata.normalize("NFKC", s).lower()
    m = DOI_CORE_RE.search(s)  # extract from URL/“doi:” etc.
    if m:
        return m.group(1)
    if s.startswith("10.") and "/" in s:
        return s
    return None

def clean_preprint_fields(df: pd.DataFrame, *, numeric_keep: int = 2, add_bucket: bool = True) -> pd.DataFrame:
    """Clean + enrich preprint fields.
    - Builds doi_prefix_first_token
    - If first token is purely numeric, keeps only its first `numeric_keep` digits
    - Optionally adds a bucketing column for grouping
    """
    df = df.drop_duplicates().copy()

    # --- gold_server_name
    df["gold_server_name"] = (
        df.get("institution_name")
          .fillna(df.get("group_title"))
          .fillna(df.get("publisher"))
    )

    # --- Normalize DOI
    if "doi" in df.columns:
        df["doi_lc"] = df["doi"].map(_normalize_doi_raw)
    else:
        df["doi_lc"] = pd.Series(pd.NA, index=df.index, dtype="object")

    # --- prefix as given
    if "prefix" in df.columns:
        df["prefix_lc"] = df["prefix"].astype(str).str.strip().str.lower()
        df.loc[df["prefix_lc"].isin(["", "nan", "none"]), "prefix_lc"] = pd.NA
    else:
        df["prefix_lc"] = pd.Series(pd.NA, index=df.index, dtype="object")

    # --- Extract prefix/suffix from normalized DOI
    doi_parts = df["doi_lc"].str.extract(r"^(10\.\d{4,9})/(.+)$")
    df["doi_prefix_from_text"] = doi_parts[0]
    df["doi_suffix"] = doi_parts[1]
    df["prefix_lc"] = df["prefix_lc"].where(df["prefix_lc"].notna(), df["doi_prefix_from_text"])

    # --- Build first segment of suffix
    starts_with_letter = df["doi_suffix"].str.match(r"^[a-z]", na=False)

    # letters-case: take only leading letters/hyphens; stop before digits or separators
    first_seg_letters = df["doi_suffix"].str.extract(
        r"^([a-z\-]+)(?=\d|[.\-_/:]|$)", expand=False
    )

    # default-case: first chunk before separators (keeps digits)
    first_seg_default = df["doi_suffix"].str.split(r"[.\-_/:\s]", n=1, regex=True).str[0]

    first_seg = pd.Series(
        np.where(starts_with_letter, first_seg_letters, first_seg_default),
        index=df.index,
        dtype="object"
    )

    # fallback: permissive token if still NA
    need_fallback = first_seg.isna() & df["doi_suffix"].notna()
    first_seg.loc[need_fallback] = df.loc[need_fallback, "doi_suffix"].str.extract(r"^([a-z0-9\-]+)", expand=False)

    # --- NEW: compress purely numeric first tokens to first `numeric_keep` digits
    if numeric_keep and numeric_keep > 0:
        numeric_only = first_seg.str.fullmatch(r"\d+", na=False)
        first_seg.loc[numeric_only] = first_seg.loc[numeric_only].str[:numeric_keep]

    # --- Assemble final token
    df["doi_prefix_first_token"] = pd.Series(pd.NA, index=df.index, dtype="object")
    ok = df["prefix_lc"].notna() & first_seg.notna() & (first_seg.astype(str) != "")
    df.loc[ok, "doi_prefix_first_token"] = df.loc[ok, "prefix_lc"].astype(str) + "/" + first_seg.loc[ok].astype(str)

    # --- Optional bucket for grouping/plots (mirrors the compressed numeric rule)
    if add_bucket:
        df["doi_prefix_bucket_2d"] = df["doi_prefix_first_token"]

    # --- Domains
    def domain_and_first_path(u):
        try:
            parsed = urlparse(str(u).lower())
            host = parsed.netloc
            if host.startswith("www."):
                host = host[4:]
            parts = re.split(r"[/=]", parsed.path)
            first_part = parts[1] if len(parts) > 1 and parts[1] else None
            return f"{host}/{first_part}" if host and first_part else (host or None)
        except Exception:
            return None

    if "landing_page_url" in df.columns:
        df["primary_domain"] = df["landing_page_url"].apply(lambda u: urlparse(str(u)).netloc.lower().replace("www.", "") if pd.notna(u) else None)
        df["primary_domain_extend"] = df["landing_page_url"].apply(domain_and_first_path)
    else:
        df["primary_domain"] = pd.Series(pd.NA, index=df.index, dtype="object")
        df["primary_domain_extend"] = pd.Series(pd.NA, index=df.index, dtype="object")

    # --- Dates → year
    if "posted_date" in df.columns:
        df["posted_date"] = pd.to_datetime(df["posted_date"], errors="coerce")
        df["year"] = df["posted_date"].dt.year

    return df

# # ----- usage
# df = clean_preprint_fields(df, numeric_keep=2, add_bucket=True)
# print(df.shape)

In [3]:
import pandas as pd
import glob
import os

def get_server_data(server_name, base_path=r"/mnt/c/SCHOLCOMMLAB/APPs/preprint-harvester/data/by_server/"):
    """
    Loads, cleans, and summarizes server metadata with clear visual formatting.
    """
    
    # 1. CONSTRUCTION & LOADING
    folder_path = os.path.join(base_path, server_name)
    parquet_files = glob.glob(os.path.join(folder_path, "*.parquet"))
    
    if not parquet_files:
        print(f"\n[!] ERROR: No parquet files found for '{server_name}'")
        print(f"    Path searched: {folder_path}\n")
        return None, None

    raw_df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)
    raw_df = raw_df.drop_duplicates('record_id')
    
    # 2. STATUS HEADER
    print("\n" + "="*50)
    print(f" SERVER ANALYSIS: {server_name.upper()}")
    print("="*50)
    print(f"  > Files found:    {len(parquet_files)}")
    print(f"  > Raw records:    {len(raw_df)}")
    
    
    # 3. CLEANING STEP
    df = clean_preprint_fields(raw_df, numeric_keep=2, add_bucket=True)
    print(f"  > Cleaned shape:  {df.shape}")
    print(f"  > Unique DOIs:    {df['doi'].nunique()}")
    print("-" * 50)


    # 4. SUMMARY GENERATION
    cols_to_summarize = [
        "doi_prefix_first_token", "primary_domain", "prefix", "member_id", 
        "publisher", "container_title", "institution_name", "group_title", 
        "issn", "type_backend_raw", "subtype_backend_raw"
    ]
    
    summary = {}
    
    for col in cols_to_summarize:
        if col in df.columns:
            # Store the data
            summary[col] = df[col].value_counts(dropna=False)
            
            # Print with plenty of space
            print(f"\nTOP VALUES FOR: {col.upper()}")
            print("-" * 30)
            print(summary[col].head(10))
            print("\n")
        else:
            summary[col] = "Column not found"
            print(f"\n[!] Column '{col}' not found in DataFrame.\n")

    print("="*50)
    print(f" END OF SUMMARY FOR {server_name.upper()}")
    print("="*50 + "\n")

    return df, summary

# ----- Execution example
# elife_df, elife_summary = get_server_data("eLife")

# Advance

In [4]:
Advance_df, Advance_summary = get_server_data("Advance")


 SERVER ANALYSIS: ADVANCE
  > Files found:    1
  > Raw records:    4401
  > Cleaned shape:  (4401, 91)
  > Unique DOIs:    4401
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31124/advance    4401
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
advance.sagepub.com    4392
authorea.com              7
techrxiv.org              2
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31124    4401
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
179    4401
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
SAGE Publications    4401
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    4401
Name: count, dtype: int64



TOP VALUES FOR: INSTITU

In [5]:
Advance_df

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,gold_server_name,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.31124/advance.7037639,Advance,crossref,10.31124/advance.7037639,10.31124/advance.7037639,https://doi.org/10.31124/advance.7037639,https://advance.sagepub.com/articles/The_Priva...,https://advance.sagepub.com/articles/The_Priva...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,The Privatization of Security and the Emergenc...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2022-03-30,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>&lt;p&gt;This study interrogates the p...,This study interrogates the participation of p...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Chinwokwu, Eke; Igbo, Emmanuel",None,None,"[{""affiliation"": [], ""family"": ""Chinwokwu"", ""g...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7037639"", ""URL"": ""ht...",SAGE Publications,10.31124/advance.7037639,10.31124,10.31124,advance.7037639,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles
1,crossref::10.31124/advance.7038500,Advance,crossref,10.31124/advance.7038500,10.31124/advance.7038500,https://doi.org/10.31124/advance.7038500,https://advance.sagepub.com/articles/Unmet_nee...,https://advance.sagepub.com/articles/Unmet_nee...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,Unmet needs of ageing transgender and non-bina...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2025-02-21,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>An view of literature on transgender a...,An view of literature on transgender and non-b...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Broadway-Horner, Matt",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8834-7...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7038500"", ""URL"": ""ht...",SAGE Publications,10.31124/advance.7038500,10.31124,10.31124,advance.7038500,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles
2,crossref::10.31124/advance.7038500.v1,Advance,crossref,10.31124/advance.7038500.v1,10.31124/advance.7038500.v1,https://doi.org/10.31124/advance.7038500.v1,https://advance

In [6]:
Advance_df.columns

Index(['record_id', 'server_name', 'backend', 'source_work_id', 'doi',
       'doi_url', 'landing_page_url', 'url_best', 'prefix', 'member_id',
       'client_id', 'provider_id', 'source_registry', 'publisher',
       'container_title', 'institution_name', 'group_title', 'issn', 'title',
       'original_title', 'short_title', 'subtitle', 'language',
       'type_backend_raw', 'subtype_backend_raw', 'type_canonical',
       'is_paratext', 'is_preprint_candidate', 'date_created', 'date_posted',
       'date_deposited', 'date_indexed', 'date_updated', 'date_issued',
       'date_registered', 'date_published', 'date_published_online',
       'publication_year', 'date_published_source', 'date_posted_source',
       'is_oa', 'oa_status', 'license', 'license_url_best', 'abstract_raw',
       'abstract_text', 'links_json_best', 'fulltext_pdf_url', 'authors_flat',
       'institutions_flat', 'countries_flat', 'authors_json',
       'contributors_json', 'editors_json', 'funders_json', 'funders_

In [7]:
df=Advance_df.copy()

In [8]:
df[df['primary_domain']=='authorea.com']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,gold_server_name,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
3980,crossref::10.31124/advance.172975247.76617035/v1,Advance,crossref,10.31124/advance.172975247.76617035/v1,10.31124/advance.172975247.76617035/v1,https://doi.org/10.31124/advance.172975247.766...,https://www.authorea.com/users/749101/articles...,https://www.authorea.com/users/749101/articles...,10.31124,179,None,None,crossref,SAGE Publications,None,Advance,Preprints,None,The Relationships between the Perception of Ph...,None,None,None,None,posted-content,preprint,preprint,None,True,2024-10-24,2024-10-24,2024-10-24,2024-10-25,None,2024-10-24,None,2024-10-24,None,2024,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,None,None,None,None,"Perlman, Amotz",0,None,"[{""affiliation"": [{""name"": ""0""}], ""family"": ""P...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.172975247.76617035/v...",Advance,10.31124/advance.172975247.76617035/v1,10.31124,10.31124,advance.172975247.76617035/v1,10.31124/advance,10.31124/advance,authorea.com,authorea.com/users
3987,crossref::10.31124/advance.173086931.17011759/v1,Advance,crossref,10.31124/advance.173086931.17011759/v1,10.31124/advance.173086931.17011759/v1,https://doi.org/10.31124/advance.173086931.170...,https://www.authorea.com/users/811231/articles...,https://www.authorea.com/users/811231/articles...,10.31124,179,None,None,crossref,SAGE Publications,None,Advance,Preprints,None,Segment-Level Traffic Volume Estimation Incorp...,None,None,None,None,posted-content,preprint,preprint,None,True,2024-11-06,2024-11-06,2024-11-06,2025-05-14,None,2024-11-06,None,2024-11-06,None,2024,issued_date,posted_date,None,None,None,None,None,None,None,None,"Morshed, Syed Ahnaf; Amine, Kamar; Hadi, Mohammed",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-3193-3...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.173086931.17011759/v...",Advance,10.31124/advance.173086931.17011759/v1,10.31124,10.31124,advance.173086931.17011759/v1,10.31124/advance,10.31124/advance,authorea.com,authorea.com/users
4029,crossref::10.31124/advance.173873930.08099093/v1,Advance,crossref,10.31124/advance.173873930.08099093/v1,10.31124/advance.173873930.08099093/v1,https://doi.org/10.31124/advance.173873930.080...,https://www.authorea.com/users/887234/articles...,https://www.authorea.com/users/887234/articles...,10.31124,179,None,None,crossref,SAGE Publications,None,Advance,Preprints,None,Defining Martial Law: Intr

In [9]:
df[df['primary_domain']=='authorea.com']['doi'].tolist()

['10.31124/advance.172975247.76617035/v1',
 '10.31124/advance.173086931.17011759/v1',
 '10.31124/advance.173873930.08099093/v1',
 '10.31124/advance.173883835.54601691/v1',
 '10.31124/advance.173892076.63256740/v1',
 '10.31124/advance.173952912.28451956/v1',
 '10.31124/advance.173883514.46750785/v1']

In [10]:
df[df['primary_domain']=='techrxiv.org']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,gold_server_name,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
3821,crossref::10.31124/advance.171330081.13982256/v1,Advance,crossref,10.31124/advance.171330081.13982256/v1,10.31124/advance.171330081.13982256/v1,https://doi.org/10.31124/advance.171330081.139...,https://www.techrxiv.org/users/678900/articles...,https://www.techrxiv.org/users/678900/articles...,10.31124,179,None,None,crossref,SAGE Publications,None,Advance,Preprints,None,Test document,None,None,None,None,posted-content,preprint,preprint,None,True,2024-04-16,2024-04-16,2024-04-16,2025-05-14,None,2024-04-16,None,2024-04-16,None,2024,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,None,None,None,None,"Meyer, Carol Anne",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-2443-2...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.171330081.13982256/v...",Advance,10.31124/advance.171330081.13982256/v1,10.31124,10.31124,advance.171330081.13982256/v1,10.31124/advance,10.31124/advance,techrxiv.org,techrxiv.org/users
3822,crossref::10.31124/advance.171330568.89328065/v1,Advance,crossref,10.31124/advance.171330568.89328065/v1,10.31124/advance.171330568.89328065/v1,https://doi.org/10.31124/advance.171330568.893...,https://www.techrxiv.org/users/678900/articles...,https://www.techrxiv.org/users/678900/articles...,10.31124,179,None,None,crossref,SAGE Publications,None,Advance,Preprints,None,This is a test preprint,None,None,None,None,posted-content,preprint,preprint,None,True,2024-04-16,2024-04-16,2024-04-16,2025-05-14,None,2024-04-16,None,2024-04-16,None,2024,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,"<jats:p id=""p1"">This is the Abstract of my tes...",This is the Abstract of my test submission,None,None,"Meyer, Carol Anne",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-2443-2...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.171330568.89328065/v...",Advance,10.31124/advance.171330568.89328065/v1,10.31124,10.31124,advance.171330568.89328065/v1,10.31124/advance,10.31124/advance,techrxiv.org,techrxiv.org/users


In [11]:
df[df['primary_domain']=='techrxiv.org']['doi'].tolist()

['10.31124/advance.171330081.13982256/v1',
 '10.31124/advance.171330568.89328065/v1']

In [12]:
df[df['institution_name']=='Authorea, Inc.']['doi'].tolist()

['10.31124/advance.24454624.v1', '10.31124/advance.170921771.12975902/v1']

In [13]:
df[df['institution_name'].isna()]

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,gold_server_name,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.31124/advance.7037639,Advance,crossref,10.31124/advance.7037639,10.31124/advance.7037639,https://doi.org/10.31124/advance.7037639,https://advance.sagepub.com/articles/The_Priva...,https://advance.sagepub.com/articles/The_Priva...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,The Privatization of Security and the Emergenc...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2022-03-30,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>&lt;p&gt;This study interrogates the p...,This study interrogates the participation of p...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Chinwokwu, Eke; Igbo, Emmanuel",None,None,"[{""affiliation"": [], ""family"": ""Chinwokwu"", ""g...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7037639"", ""URL"": ""ht...",SAGE Publications,10.31124/advance.7037639,10.31124,10.31124,advance.7037639,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles
1,crossref::10.31124/advance.7038500,Advance,crossref,10.31124/advance.7038500,10.31124/advance.7038500,https://doi.org/10.31124/advance.7038500,https://advance.sagepub.com/articles/Unmet_nee...,https://advance.sagepub.com/articles/Unmet_nee...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,Unmet needs of ageing transgender and non-bina...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2025-02-21,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>An view of literature on transgender a...,An view of literature on transgender and non-b...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Broadway-Horner, Matt",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8834-7...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7038500"", ""URL"": ""ht...",SAGE Publications,10.31124/advance.7038500,10.31124,10.31124,advance.7038500,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles
2,crossref::10.31124/advance.7038500.v1,Advance,crossref,10.31124/advance.7038500.v1,10.31124/advance.7038500.v1,https://doi.org/10.31124/advance.7038500.v1,https://advance

In [14]:
df[df['institution_name'].isna()]['landing_page_url'].tolist()

['https://advance.sagepub.com/articles/The_Privatization_of_Security_and_the_Emergence_of_Private_Security_Companies_in_Crime_Control_in_Nigeria_docx/7037639',
 'https://advance.sagepub.com/articles/Unmet_needs_of_ageing_transgender_and_non-binary_population_An_Overview/7038500',
 'https://advance.sagepub.com/articles/Unmet_needs_of_ageing_transgender_and_non-binary_population_An_Overview/7038500/1',
 'https://advance.sagepub.com/articles/A_TROG_masked_docUsing_T_R_O_G_to_improve_psychotherapy_engagement_when_working_with_Mild_Learning_Disability_Populations/7038503',
 'https://advance.sagepub.com/articles/A_TROG_masked_docUsing_T_R_O_G_to_improve_psychotherapy_engagement_when_working_with_Mild_Learning_Disability_Populations/7038503/1',
 'https://advance.sagepub.com/articles/Are_We_There_Yet_Understanding_Cultural_Issues_and_Making_them_Central_in_Psychology_and_Psychiatry/7038506/1',
 'https://advance.sagepub.com/articles/Are_We_There_Yet_Understanding_Cultural_Issues_and_Making_them

In [15]:
df[df['institution_name']=='Advance']['landing_page_url'].tolist()

['https://advance.sagepub.com/articles/preprint/Stress_Scale_in_the_Context_of_Online_Learning_among_Junior_High_School_Students_ages_11-17_Development_Validity_and_Reliability/15020103',
 'https://advance.sagepub.com/articles/preprint/Gifted_Programming_Identification_Procedures_A_Hidden_Curriculum/15040629',
 'https://advance.sagepub.com/articles/preprint/Gendered_Justice_The_Impact_of_Gender_on_Criminal_Justice_Policies_and_Legislation_throughout_the_United_Kingdom/15042804',
 'https://advance.sagepub.com/articles/preprint/Maximization_of_Female_Contribution_in_Global_Workforce/15043020',
 'https://advance.sagepub.com/articles/preprint/_Only_I_have_to_help_myself_Indian_Migrant_Workers_Plight_During_COVID-19_Lockdown/15047967',
 'https://advance.sagepub.com/articles/preprint/_Only_I_have_to_help_myself_Indian_Migrant_Workers_Plight_During_COVID-19_Lockdown/15047967/1',
 'https://advance.sagepub.com/articles/preprint/Changes_in_General_and_Specific_Teacher_Self-Efficacy_Related_to_Pr

In [16]:
df['doi'].sort_values().head(60)#.tolist()

354        10.31124/advance.10005662
1779    10.31124/advance.10005662.v1
1780    10.31124/advance.10005662.v2
340        10.31124/advance.10005884
1781    10.31124/advance.10005884.v1
346        10.31124/advance.10007411
1782    10.31124/advance.10007411.v1
342        10.31124/advance.10010381
1783    10.31124/advance.10010381.v1
343        10.31124/advance.10012031
1785    10.31124/advance.10012031.v1
355        10.31124/advance.10026860
1784    10.31124/advance.10026860.v1
390        10.31124/advance.10048160
1768    10.31124/advance.10048160.v1
1787    10.31124/advance.10048160.v2
1786    10.31124/advance.10048160.v3
3715    10.31124/advance.10048160.v4
653        10.31124/advance.10050131
1789    10.31124/advance.10050131.v1
345        10.31124/advance.10050497
1788    10.31124/advance.10050497.v1
347        10.31124/advance.10055363
1790    10.31124/advance.10055363.v1
344        10.31124/advance.10055399
1791    10.31124/advance.10055399.v1
348        10.31124/advance.10096058
1

In [17]:
df[df['subtype_backend_raw']=='other']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,gold_server_name,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
12,crossref::10.31124/advance.7038497,Advance,crossref,10.31124/advance.7038497,10.31124/advance.7038497,https://doi.org/10.31124/advance.7038497,https://advance.sagepub.com/articles/A_Snapsho...,https://advance.sagepub.com/articles/A_Snapsho...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,A Snapshot of Psycho-social issues in Camp Liv...,None,None,None,None,posted-content,other,other,None,True,2018-09-06,2018-09-06,2018-09-06,2025-02-21,None,2018-09-06,None,2018-09-06,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>Using survey data to measure the impac...,Using survey data to measure the impact of a d...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Broadway-Horner, Matt",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8834-7...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7038497"", ""URL"": ""ht...",SAGE Publications,10.31124/advance.7038497,10.31124,10.31124,advance.7038497,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles
13,crossref::10.31124/advance.7038497.v1,Advance,crossref,10.31124/advance.7038497.v1,10.31124/advance.7038497.v1,https://doi.org/10.31124/advance.7038497.v1,https://advance.sagepub.com/articles/A_Snapsho...,https://advance.sagepub.com/articles/A_Snapsho...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,A Snapshot of Psycho-social issues in Camp Liv...,None,None,None,None,posted-content,other,other,None,True,2018-09-06,2018-09-06,2018-09-06,2025-02-21,None,2018-09-06,None,2018-09-06,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>Using survey data to measure the impac...,Using survey data to measure the impact of a d...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Broadway-Horner, Matt",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8834-7...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7038497.v1"", ""URL"": ...",SAGE Publications,10.31124/advance.7038497.v1,10.31124,10.31124,advance.7038497.v1,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles


In [18]:
df[df['subtype_backend_raw']=='other']['doi'].tolist()

['10.31124/advance.7038497', '10.31124/advance.7038497.v1']

## observation

it seems like their create the new doi with versions patterns and the old records without version patterns are not available on the main page

In [19]:
df['doi'].sort_values().head(60)#.tolist()


354        10.31124/advance.10005662
1779    10.31124/advance.10005662.v1
1780    10.31124/advance.10005662.v2
340        10.31124/advance.10005884
1781    10.31124/advance.10005884.v1
346        10.31124/advance.10007411
1782    10.31124/advance.10007411.v1
342        10.31124/advance.10010381
1783    10.31124/advance.10010381.v1
343        10.31124/advance.10012031
1785    10.31124/advance.10012031.v1
355        10.31124/advance.10026860
1784    10.31124/advance.10026860.v1
390        10.31124/advance.10048160
1768    10.31124/advance.10048160.v1
1787    10.31124/advance.10048160.v2
1786    10.31124/advance.10048160.v3
3715    10.31124/advance.10048160.v4
653        10.31124/advance.10050131
1789    10.31124/advance.10050131.v1
345        10.31124/advance.10050497
1788    10.31124/advance.10050497.v1
347        10.31124/advance.10055363
1790    10.31124/advance.10055363.v1
344        10.31124/advance.10055399
1791    10.31124/advance.10055399.v1
348        10.31124/advance.10096058
1

In [20]:
# [./] matches either a dot or a slash
# v\d+ matches 'v' followed by one or more digits
# $ ensures this pattern is at the very end of the string
pattern = r'[./]v\d+$'
# pattern = r'v\d+$'
mask = df['doi'].str.contains(pattern, regex=True, na=False)
result = df[mask]
result

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,gold_server_name,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
2,crossref::10.31124/advance.7038500.v1,Advance,crossref,10.31124/advance.7038500.v1,10.31124/advance.7038500.v1,https://doi.org/10.31124/advance.7038500.v1,https://advance.sagepub.com/articles/Unmet_nee...,https://advance.sagepub.com/articles/Unmet_nee...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,Unmet needs of ageing transgender and non-bina...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2025-02-21,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>An view of literature on transgender a...,An view of literature on transgender and non-b...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Broadway-Horner, Matt",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8834-7...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7038500.v1"", ""URL"": ...",SAGE Publications,10.31124/advance.7038500.v1,10.31124,10.31124,advance.7038500.v1,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles
4,crossref::10.31124/advance.7038503.v1,Advance,crossref,10.31124/advance.7038503.v1,10.31124/advance.7038503.v1,https://doi.org/10.31124/advance.7038503.v1,https://advance.sagepub.com/articles/A_TROG_ma...,https://advance.sagepub.com/articles/A_TROG_ma...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,A TROG masked.docUsing T.R.O.G to improve psyc...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2025-02-21,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>A discussion on language from a theore...,A discussion on language from a theoretical pe...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Broadway-Horner, Matt",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8834-7...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7038503.v1"", ""URL"": ...",SAGE Publications,10.31124/advance.7038503.v1,10.31124,10.31124,advance.7038503.v1,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles
5,crossref::10.31124/advance.7038506.v1,Advance,crossref,10.31124/advance.7038506.v1,10.31124/advance.7038506.v1,https://doi.org/10.31124/advance.70385

In [21]:
# [./] matches either a dot or a slash
# v\d+ matches 'v' followed by one or more digits
# $ ensures this pattern is at the very end of the string
pattern = r'[./]v\d+$'
# pattern = r'v\d+$'
mask = ~df['doi'].str.contains(pattern, regex=True, na=False)
result = df[mask]
result

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,gold_server_name,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.31124/advance.7037639,Advance,crossref,10.31124/advance.7037639,10.31124/advance.7037639,https://doi.org/10.31124/advance.7037639,https://advance.sagepub.com/articles/The_Priva...,https://advance.sagepub.com/articles/The_Priva...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,The Privatization of Security and the Emergenc...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2022-03-30,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>&lt;p&gt;This study interrogates the p...,This study interrogates the participation of p...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Chinwokwu, Eke; Igbo, Emmanuel",None,None,"[{""affiliation"": [], ""family"": ""Chinwokwu"", ""g...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7037639"", ""URL"": ""ht...",SAGE Publications,10.31124/advance.7037639,10.31124,10.31124,advance.7037639,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles
1,crossref::10.31124/advance.7038500,Advance,crossref,10.31124/advance.7038500,10.31124/advance.7038500,https://doi.org/10.31124/advance.7038500,https://advance.sagepub.com/articles/Unmet_nee...,https://advance.sagepub.com/articles/Unmet_nee...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,Unmet needs of ageing transgender and non-bina...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2025-02-21,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>An view of literature on transgender a...,An view of literature on transgender and non-b...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Broadway-Horner, Matt",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8834-7...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7038500"", ""URL"": ""ht...",SAGE Publications,10.31124/advance.7038500,10.31124,10.31124,advance.7038500,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/articles
3,crossref::10.31124/advance.7038503,Advance,crossref,10.31124/advance.7038503,10.31124/advance.7038503,https://doi.org/10.31124/advance.7038503,https://advance.sagepub.com

In [22]:
pattern = "10.31124/advance.14132306"

mask = df['doi'].str.contains(pattern, regex=False, na=False)
result = df[mask]
result

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,gold_server_name,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
2473,crossref::10.31124/advance.14132306,Advance,crossref,10.31124/advance.14132306,10.31124/advance.14132306,https://doi.org/10.31124/advance.14132306,https://advance.sagepub.com/doi/full/10.31124/...,https://advance.sagepub.com/doi/full/10.31124/...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,The Effect of Healthcare Education on Future D...,None,None,None,None,posted-content,preprint,preprint,None,True,2021-03-04,2021-03-09,2024-02-22,2024-07-17,None,2021-03-09,None,2021-03-09,None,2021,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>&lt;p&gt;In competitive education test...,In competitive education test scores and scien...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Nowak, Ewa; Barciszewska, Anna-Maria; Lind, Ge...",None,None,"[{""affiliation"": [], ""family"": ""Nowak"", ""given...",None,None,None,None,None,None,None,None,1,None,None,1,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.14132306"", ""URL"": ""h...",SAGE Publications,10.31124/advance.14132306,10.31124,10.31124,advance.14132306,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/doi
2475,crossref::10.31124/advance.14132306.v1,Advance,crossref,10.31124/advance.14132306.v1,10.31124/advance.14132306.v1,https://doi.org/10.31124/advance.14132306.v1,https://advance.sagepub.com/doi/full/10.31124/...,https://advance.sagepub.com/doi/full/10.31124/...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,The effect of healthcare education on students...,None,None,None,None,posted-content,preprint,preprint,None,True,2021-03-04,2021-03-04,2024-02-22,2024-02-23,None,2021-03-04,None,2021-03-04,None,2021,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>&lt;p&gt;In competitive education test...,In competitive education test scores and scien...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Nowak, Ewa; Barciszewska, Anna-Maria; Lind, Ge...",None,None,"[{""affiliation"": [], ""family"": ""Nowak"", ""given...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.14132306.v1"", ""URL"":...",SAGE Publications,10.31124/advance.14132306.v1,10.31124,10.31124,advance.14132306.v1,10.31124/advance,10.31124/advance,advance.sagepub.com,advance.sagepub.com/doi
2477,crossref::10.31124/advance.14132306.v2,Advance,crossref,10.31124/advance.14132306.v2,10.311

# AfricArXiv

In [23]:
AfricArXiv_df, AfricArXiv_summary = get_server_data("AfricArXiv")


 SERVER ANALYSIS: AFRICARXIV
  > Files found:    2
  > Raw records:    2190
  > Cleaned shape:  (2190, 91)
  > Unique DOIs:    2190
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.60763/africarxiv    1689
10.31730/osf            496
10.31235/osf              5
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
africarxiv.ubuntunet.net    1689
osf.io                       501
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.60763    1689
10.31730     496
10.31235       5
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None     1689
15934     501
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
My University                                                  1423
Center for Open Science              

# AgEcon Search

In [4]:
AgEcon_Search_df, AgEcon_Search_summary = get_server_data("AgEcon_Search")


 SERVER ANALYSIS: AGECON_SEARCH
  > Files found:    2
  > Raw records:    188173
  > Cleaned shape:  (188173, 91)
  > Unique DOIs:    188173
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.22004/ag      188172
10.22004/tind         1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
ageconsearch.umn.edu    188173
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.22004    188173
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    188173
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Unknown    188173
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    188173
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
---------

In [8]:
for x in AgEcon_Search_df.sample(1).iloc[0]:
    print(x)

datacite::10.22004/ag.econ.138022
AgEcon Search
datacite
10.22004/ag.econ.138022
10.22004/ag.econ.138022
https://doi.org/10.22004/ag.econ.138022
https://ageconsearch.umn.edu/record/138022
https://ageconsearch.umn.edu/record/138022
10.22004
None
tind.agecon
tind
datacite
Unknown
None
None
None
None
State-Level Output Supply and Input Demand Elasticities for Agricultural Commodities
None
None
None
en
Text
Text
Text
None
None
2019-08-30
None
None
None
2020-07-29
None
2019-08-30
None
None
1992
published_year
None
None
None
None
None
[{"description": "Own- and cross-price production elasticities, estimated in four major agricultural states (California, Iowa, Texas, and Florida), measure the sensitivity to price changes of as many as 25 individual crop and livestock output supplies and six input demands. While most responses were highly inelastic, a wide range of elasticities occurred across States. The range was generally greater for crop supplies than for livestock supplies or input demand

# AgriRxiv

In [25]:
AgriRxiv_df, AgriRxiv_summary = get_server_data("AgriRxiv")


 SERVER ANALYSIS: AGRIRXIV
  > Files found:    1
  > Raw records:    818
  > Cleaned shape:  (818, 91)
  > Unique DOIs:    818
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31220/osf         391
10.31220/agrirxiv    380
10.31227/osf          22
10.31219/osf          10
10.31235/osf           9
10.31234/osf           5
10.31225/osf           1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
cabidigitallibrary.org    462
osf.io                    356
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31220    771
10.31227     22
10.31219     10
10.31235      9
10.31234      5
10.31225      1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
242      771
15934     47
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
-----------------------

# AIJR Preprints

In [26]:
AIJR_Preprints_df, AIJR_Preprints_summary = get_server_data("AIJR_Preprints")


 SERVER ANALYSIS: AIJR_PREPRINTS
  > Files found:    1
  > Raw records:    143
  > Cleaned shape:  (143, 91)
  > Unique DOIs:    143
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.21467/preprints    143
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprints.aijr.org    143
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.21467    143
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
8901    143
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
AIJR Publisher    143
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    143
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
N

# AMRC Open Research

In [27]:
AMRC_Open_Research_df, AMRC_Open_Research_summary = get_server_data("AMRC_Open_Research")


 SERVER ANALYSIS: AMRC_OPEN_RESEARCH
  > Files found:    2
  > Raw records:    102
  > Cleaned shape:  (102, 91)
  > Unique DOIs:    102
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12688/amrcopenres      52
10.12688/healthopenres    50
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
healthopenresearch.org    62
amrcopenresearch.org      40
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12688    102
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2560    102
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
F1000 Research Ltd    102
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
Health Open Research                             61

In [28]:
AMRC_Open_Research_df.sort_values(by='title')

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,gold_server_name,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
98,crossref::10.12688/healthopenres.13924.1,AMRC Open Research,crossref,10.12688/healthopenres.13924.1,10.12688/healthopenres.13924.1,https://doi.org/10.12688/healthopenres.13924.1,https://healthopenresearch.org/articles/7-17/v1,https://healthopenresearch.org/articles/7-17/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Health Open Research,None,None,2753-6416,A Protocol for Systematic Review of Prognostic...,None,None,None,en,journal-article,None,journal-article,None,False,2025-11-21,None,2025-11-21,2025-11-21,None,2025-10-10,None,2025-10-10,2025-10-10,2025.0,issued_date,None,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<ns3:p>Background Survivors of adolescent and ...,Background Survivors of adolescent and young a...,"[{""URL"": ""https://healthopenresearch.org/artic...",https://healthopenresearch.org/articles/7-17/v...,"Guolla, Louise; Mbuagbaw, Lawrence; Ma, Jinhui...",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-2199-6...",None,None,"[{""award"": [""CVG-186971""], ""award-info"": [{""aw...",CIHR Vanier Canada Graduate Scholarship,1.0,None,None,None,0,None,None,0,58,"[{""DOI"": ""10.1161/CIRCULATIONAHA.119.041403"", ...",None,,,False,None,,None,None,,None,None,https://doi.org/10.12688/healthopenres.crossma...,issn,0,None,"{""DOI"": ""10.12688/healthopenres.13924.1"", ""ISS...",F1000 Research Ltd,10.12688/healthopenres.13924.1,10.12688,10.12688,healthopenres.13924.1,10.12688/healthopenres,10.12688/healthopenres,healthopenresearch.org,healthopenresearch.org/articles
17,crossref::10.12688/amrcopenres.12936.2,AMRC Open Research,crossref,10.12688/amrcopenres.12936.2,10.12688/amrcopenres.12936.2,https://doi.org/10.12688/amrcopenres.12936.2,https://amrcopenresearch.org/articles/2-29/v2,https://amrcopenresearch.org/articles/2-29/v2,10.12688,2560,None,None,crossref,F1000 Research Ltd,AMRC Open Research,None,None,2517-6900,A collaborative approach to exercise provision...,None,None,None,en,journal-article,None,journal-article,None,False,2021-04-01,None,2021-04-26,2026-02-27,None,2021-04-01,None,2021-04-01,2021-04-01,2021.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns3:p>\n <ns3:bold>Backgro...,Background: Exercise has been shown to be bene...,"[{""URL"": ""https://amrcopenresearch.org/article...",https://amrcopenresearch.org/articles/2-29/v2/pdf,"Jones, Julie; Alexander, Lyndsay; Hancock, Eli...",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-1943-1...",None,None,"[{""award"": [""F-1901""], ""award-info"": [{""award-...","Parkinson's UK; Chief Scientist Office, Scotland",2.0,None,None,None,1,

# APSA Preprints

In [29]:
APSA_Preprints_df, APSA_Preprints_summary = get_server_data("APSA_Preprints")


 SERVER ANALYSIS: APSA_PREPRINTS
  > Files found:    1
  > Raw records:    1470
  > Cleaned shape:  (1470, 91)
  > Unique DOIs:    1470
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.33774/apsa-    1470
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprints.apsanet.org    1470
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.33774    1470
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
56    1470
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Cambridge University Press (CUP)    1470
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    1470
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
-------------------------

# Arabixiv

In [30]:
Arabixiv_df, Arabixiv_summary = get_server_data("Arabixiv")


 SERVER ANALYSIS: ARABIXIV
  > Files found:    1
  > Raw records:    502
  > Cleaned shape:  (502, 91)
  > Unique DOIs:    502
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31221/osf    412
10.31219/osf     34
10.31227/osf     25
10.31234/osf     14
10.31235/osf      6
10.31223/osf      5
10.31220/osf      3
10.31228/osf      2
10.31225/osf      1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    502
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31221    412
10.31219     34
10.31227     25
10.31234     14
10.31235      6
10.31223      5
10.31220      3
10.31228      2
10.31225      1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    494
29705      5
242        3
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
---

# ARPHA Preprints

In [31]:
ARPHA_df, ARPHA_summary = get_server_data("ARPHA_Preprints")


 SERVER ANALYSIS: ARPHA_PREPRINTS
  > Files found:    1
  > Raw records:    890
  > Cleaned shape:  (890, 91)
  > Unique DOIs:    890
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.3897/arphapreprints    890
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprints.arphahub.com    890
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.3897    890
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2258    890
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Pensoft Publishers    890
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    890
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
instit

# ART-Dok

In [32]:
ART_df, ART_summary = get_server_data("ART-Dok")


 SERVER ANALYSIS: ART-DOK
  > Files found:    2
  > Raw records:    9653
  > Cleaned shape:  (9653, 91)
  > Unique DOIs:    9653
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.11588/artdok    9653
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
archiv.ub.uni-heidelberg.de    9651
ub.uni-heidelberg.de              2
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.11588    9653
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    9653
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Heidelberg University Library        9644
None                                    5
Vandenhoeck & Ruprecht                  1
Muzeum Uniwersytetu Warszawskiego       1
Biblioteka Jagiellońska                 1
D

# arXiv

In [33]:
# arXiv_df, arXiv_summary = get_server_data("arXiv")

# Authorea Inc.

In [34]:
Authorea_df, Authorea_summary = get_server_data("Authorea_Inc.")


 SERVER ANALYSIS: AUTHOREA_INC.
  > Files found:    1
  > Raw records:    65450
  > Cleaned shape:  (65450, 91)
  > Unique DOIs:    65450
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.22541/au          61998
10.22541/essoar       3325
10.1002/essoar         105
10.22541/21docs         17
10.22541/techrxiv        2
10.31124/advance         2
10.15200/winn            1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
authorea.com                    61424
essopenarchive.org               3594
techrxiv.org                      374
advance.sagepub.com                46
journal.sketchingscience.org        4
21docs.com                          4
cise@computer.org                   3
doi.org                             1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.22541    65342
10.1002  

# Beilstein Archives

In [35]:
Beilstein_df, Beilstein_summary = get_server_data("Beilstein_Archives")


 SERVER ANALYSIS: BEILSTEIN_ARCHIVES
  > Files found:    1
  > Raw records:    697
  > Cleaned shape:  (697, 91)
  > Unique DOIs:    697
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.3762/bxiv    697
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
beilstein-archives.org    697
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.3762    697
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2086    697
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Beilstein Institut    697
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    697
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_n

# BioHackrXiv

In [36]:
BioHackrXiv_df, BioHackrXiv_summary = get_server_data("BioHackrXiv")


 SERVER ANALYSIS: BIOHACKRXIV
  > Files found:    1
  > Raw records:    139
  > Cleaned shape:  (139, 91)
  > Unique DOIs:    139
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.37044/osf    139
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    139
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.37044    139
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    139
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Open Science    139
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    139
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None    139


# bioRxiv

In [37]:
bioRxiv_df, bioRxiv_summary = get_server_data("bioRxiv")


 SERVER ANALYSIS: BIORXIV
  > Files found:    1
  > Raw records:    306948
  > Cleaned shape:  (306948, 91)
  > Unique DOIs:    306948
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.1101/20    240359
10.1101/10       828
10.1101/32       809
10.1101/19       806
10.1101/42       805
10.1101/58       802
10.1101/17       802
10.1101/09       802
10.1101/45       800
10.1101/14       798
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
biorxiv.org    306948
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.1101    306948
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
246    306948
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Cold Spring Harbor Laboratory    306948
Name: count, dtype: int64



# BodoArXiv

In [38]:
BodoArXiv_df, BodoArXiv_summary = get_server_data("BodoArXiv")


 SERVER ANALYSIS: BODOARXIV
  > Files found:    1
  > Raw records:    165
  > Cleaned shape:  (165, 91)
  > Unique DOIs:    165
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.34055/osf    162
10.31219/osf      3
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    165
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.34055    162
10.31219      3
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    165
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Open Science    165
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    165
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
--------------------------

# Cambridge Open Engage

In [39]:
Cambridge_df, Cambridge_summary = get_server_data("Cambridge_Open_Engage")


 SERVER ANALYSIS: CAMBRIDGE_OPEN_ENGAGE
  > Files found:    1
  > Raw records:    3090
  > Cleaned shape:  (3090, 91)
  > Unique DOIs:    3090
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.33774/coe-    3090
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
cambridge.org    3090
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.33774    3090
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
56    3090
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Cambridge University Press (CUP)    3090
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    3090
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
---------------------------

# CERN document server

In [40]:
CERN_df, CERN_summary = get_server_data("CERN_document_server")


 SERVER ANALYSIS: CERN_DOCUMENT_SERVER
  > Files found:    2
  > Raw records:    973
  > Cleaned shape:  (973, 91)
  > Unique DOIs:    973
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.17181/cern    49
10.17181/d       16
10.17181/c       15
10.17181/q       15
10.17181/n       14
10.17181/m       13
10.17181/s       12
10.17181/h       11
10.17181/z       10
10.17181/v        9
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
repository.cern    926
cds.cern.ch         46
new-cds.cern.ch      1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.17181    973
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    973
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
CERN                          

/tmp/ipykernel_53666/1196196344.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  raw_df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)


# ChemRxiv

In [41]:
ChemRxiv_df, ChemRxiv_summary = get_server_data("ChemRxiv")


 SERVER ANALYSIS: CHEMRXIV
  > Files found:    1
  > Raw records:    46475
  > Cleaned shape:  (46475, 91)
  > Unique DOIs:    46475
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.26434/chemrxiv-    35040
10.26434/chemrxiv     11435
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
chemrxiv.org    46475
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.26434    46475
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
316    46475
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
American Chemical Society (ACS)    46475
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    46475
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME

# CogPrints

In [76]:
CogPrints_df, CogPrints_summary = get_server_data("CogPrints")


 SERVER ANALYSIS: COGPRINTS
  > Files found:    1
  > Raw records:    1537
  > Cleaned shape:  (1537, 91)
  > Unique DOIs:    39
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>               1498
10.48550/arxiv       14
10.5281/zenodo        7
10.13140/rg           7
10.13140/2            2
10.60692/cs           1
10.6084/m             1
10.60692/frxsy-       1
10.60692/ny           1
10.48456/tr-          1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
None             1041
cogprints.org     471
doi.org            25
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
None        1498
10.48550      14
10.13140       9
10.5281        7
10.60692       4
10.6084        1
10.48456       1
10.71910       1
10.18267       1
10.13016       1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_I

In [77]:
CogPrints_df

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,gold_server_name,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,openalex::W1555476679,CogPrints,openalex,https://openalex.org/W1555476679,None,None,http://cogprints.org/418/2/context3.ps,http://cogprints.org/418/2/context3.ps,None,None,None,None,openalex,None,None,None,None,None,Notes on formalizing context,None,None,None,en,article,None,article,False,False,2025-10-10T00:00:00,None,None,None,2025-10-10T17:16:08.811792,None,None,1993-08-28,None,1993,openalex.publication_date,None,False,closed,None,None,"{""1"": [127], ""AI"": [62, 97], ""Fully"": [77], ""I...",These notes discuss formalizing contexts as fi...,None,None,John McCarthy,Stanford University,US,"[{""affiliations"": [{""institution_ids"": [""https...",None,None,[],None,NaN,None,"[{""display_name"": ""Computer science"", ""id"": ""h...","[{""display_name"": ""Logic, Reasoning, and Knowl...",764,None,764,None,9,"[""https://openalex.org/W2138162238"", ""https://...",None,None,None,None,None,None,None,None,None,None,None,None,source_id,5,None,"{""abstract_inverted_index"": {""1"": [127], ""AI"":...",None,None,NaN,NaN,NaN,<NA>,<NA>,cogprints.org,cogprints.org/418
1,openalex::W2078579128,CogPrints,openalex,https://openalex.org/W2078579128,None,None,None,None,None,None,None,None,openalex,None,None,None,None,None,Solving the multiple-instance problem: A lazy ...,None,None,None,en,preprint,None,preprint,False,True,2025-10-10T00:00:00,None,None,None,2025-11-06T04:12:42.849631,None,None,2009-03-22,None,2009,openalex.publication_date,None,True,green,None,None,"{""1."": [116], ""As"": [0], ""Bayesian-KNN"": [72],...","As opposed to traditional supervised learning,...",None,http://cogprints.org/2124/3/wang_ICML2000.pdf,Jun Wang; Jean‐Daniel Zucker,University of Illinois Urbana-Champaign,US,"[{""affiliations"": [{""institution_ids"": [""https...",None,None,[],None,NaN,None,"[{""display_name"": ""Computer science"", ""id"": ""h...","[{""display_name"": ""Image Retrieval and Classif...",577,None,577,None,22,"[""https://openalex.org/W2911678770"", ""https://...",None,None,None,None,None,None,None,None,None,None,None,None,source_id,5,None,"{""abstract_inverted_index"": {""1."": [116], ""As""...",None,None,NaN,NaN,NaN,<NA>,<NA>,None,None
2,openalex::W1556724924,CogPrints,openalex,https://openalex.org/W1556724924,None,None,None,None,None,None,None,None,openalex,None,None,None,None,None,A Unifying Field in Logics: Neutrosophic Logic.,None,None,None,en,book-chapter,None,book-chapter,False,False,2025-10-10T00:00:00,None,None,None,2025-11-06T04:12:42.849631,None,None,1999-01-01,None,1999,openalex.publication_date,None,True,green,None,None,"{""Also,"": [62], ""Similarly,"": [53], ""The"": [0]...",The author makes an introduction to non-standa...,None,http://cogpr

# CoP

In [43]:
CoP_df, CoP_summary = get_server_data("CoP")


 SERVER ANALYSIS: COP
  > Files found:    1
  > Raw records:    30
  > Cleaned shape:  (30, 91)
  > Unique DOIs:    30
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31219/osf    30
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    30
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31219    30
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    30
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Open Science    30
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    30
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None    30
Name: count, dtype

# Covid-19 Preprints

In [44]:
Covid_df, Covid_summary = get_server_data("Covid-19_Preprints")


 SERVER ANALYSIS: COVID-19_PREPRINTS
  > Files found:    1
  > Raw records:    647
  > Cleaned shape:  (647, 91)
  > Unique DOIs:    647
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.21055/preprints-    647
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
covid19-preprints.microbe.ru    647
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.21055    647
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
8634    647
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Russian Research Anti-Plague Institute Microbe    647
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    647
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
--

# CrimRxiv

In [78]:
CrimRxiv_df, CrimRxiv_summary = get_server_data("CrimRxiv")


 SERVER ANALYSIS: CRIMRXIV
  > Files found:    1
  > Raw records:    2838
  > Cleaned shape:  (2838, 91)
  > Unique DOIs:    2838
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.21428/cb          2827
10.21428/51bae76e      11
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
crimrxiv.com           2806
crimrxiv.pubpub.org      21
oqc.crimrxiv.com         11
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.21428    2838
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
9621    2838
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
PubPub    2838
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
CrimRxiv                        2678
None      

In [79]:
CrimRxiv_df

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,gold_server_name,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.21428/cb6ab371.30868967,CrimRxiv,crossref,10.21428/cb6ab371.30868967,10.21428/cb6ab371.30868967,https://doi.org/10.21428/cb6ab371.30868967,https://crimrxiv.pubpub.org/pub/tn1vo8l2,https://crimrxiv.pubpub.org/pub/tn1vo8l2,10.21428,9621,None,None,crossref,PubPub,CrimRxiv,None,None,None,"Bentham, Not Epicurus: The Relevance of Pleasu...",None,None,None,en,journal-article,None,journal-article,None,False,2020-07-07,None,2020-07-07,2025-11-23,None,2020-07-07,None,2020-07-07,2020-07-07,2020.0,issued_date,None,None,None,None,None,None,None,None,None,"Jacques, Scott",Georgia State University,None,"[{""affiliation"": [{""name"": ""Georgia State Univ...",None,None,None,None,None,None,None,None,1,None,None,1,0,None,None,,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,25,None,"{""DOI"": ""10.21428/cb6ab371.30868967"", ""URL"": ""...",PubPub,10.21428/cb6ab371.30868967,10.21428,10.21428,cb6ab371.30868967,10.21428/cb,10.21428/cb,crimrxiv.pubpub.org,crimrxiv.pubpub.org/pub
1,crossref::10.21428/cb6ab371.aab5ffbe,CrimRxiv,crossref,10.21428/cb6ab371.aab5ffbe,10.21428/cb6ab371.aab5ffbe,https://doi.org/10.21428/cb6ab371.aab5ffbe,https://crimrxiv.pubpub.org/pub/oxbqg1op,https://crimrxiv.pubpub.org/pub/oxbqg1op,10.21428,9621,None,None,crossref,PubPub,CrimRxiv,None,None,None,Proterrence &amp; Rule Illegitimacy in an Age ...,None,None,None,en,journal-article,None,journal-article,None,False,2020-07-07,None,2020-07-07,2022-04-05,None,2020-07-07,None,2020-07-07,2020-07-07,2020.0,issued_date,None,None,None,None,None,None,None,None,None,"Jacobs, Bruce; Jacques, Scott","University of Texas, Dallas; Georgia State Uni...",None,"[{""affiliation"": [{""name"": ""University of Texa...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,25,None,"{""DOI"": ""10.21428/cb6ab371.aab5ffbe"", ""URL"": ""...",PubPub,10.21428/cb6ab371.aab5ffbe,10.21428,10.21428,cb6ab371.aab5ffbe,10.21428/cb,10.21428/cb,crimrxiv.pubpub.org,crimrxiv.pubpub.org/pub
2,crossref::10.21428/cb6ab371.9e0bdd09,CrimRxiv,crossref,10.21428/cb6ab371.9e0bdd09,10.21428/cb6ab371.9e0bdd09,https://doi.org/10.21428/cb6ab371.9e0bdd09,https://crimrxiv.pubpub.org/pub/zk9k26ba,https://crimrxiv.pubpub.org/pub/zk9k26ba,10.21428,9621,None,None,crossref,PubPub,CrimRxiv,None,None,None,La cybercriminalité,None,None,None,en,journal-article,None,journal-article,None,False,2020-07-08,None,2020-07-08,2023-08-16,None,2020-07-08,None,2020-07-08,2020-07-08,2020.0,issued_date,None,None,None,None,None,None,None,None,None,"Décary-Hétu, David",None,None,"[{""affiliation"": [], ""family"": ""Décary-Hétu"", ...",None,None,None,None,None,

# CrossAsia-Repository

In [46]:
CrossAsia_df, CrossAsia_summary = get_server_data("CrossAsia-Repository")


 SERVER ANALYSIS: CROSSASIA-REPOSITORY
  > Files found:    2
  > Raw records:    479
  > Cleaned shape:  (479, 91)
  > Unique DOIs:    479
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.48796/20    479
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
repository.crossasia.org    479
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.48796    479
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    479
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Fachinformationsdienst (FID) Asien                                                             428
CrossAsia-eBooks                                                                                21
Iudicium                                          

# Digital Access to Scholarship at Harvard (DASH) (Harvard University)

In [80]:
DASH_df, DASH_summary = get_server_data("Digital_Access_to_Scholarship_at_Harvard_(DASH)_(H")


 SERVER ANALYSIS: DIGITAL_ACCESS_TO_SCHOLARSHIP_AT_HARVARD_(DASH)_(H
  > Files found:    1
  > Raw records:    9703
  > Cleaned shape:  (9703, 91)
  > Unique DOIs:    154
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>              9549
10.48550/arxiv      65
10.53901/tjbs        4
10.7916/d            4
10.15779/z           3
10.1901/jaba         2
10.1257/aer          2
10.1186/14           2
10.13140/rg          2
10.7448/ias          2
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
nrs.harvard.edu               8768
dash.harvard.edu               918
harvardlibrarybulletin.org       6
dissertations.umi.com            4
doi.org                          1
globalasia.org                   1
frstrategie.org                  1
iase-web.org                     1
ipres2023.us                     1
historytoday.com                

# DSpace@MIT

In [48]:
DSpace_df, DSpace_summary = get_server_data("DSpace@MIT")


 SERVER ANALYSIS: DSPACE@MIT
  > Files found:    1
  > Raw records:    12661
  > Cleaned shape:  (12661, 91)
  > Unique DOIs:    1308
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>               11353
10.48550/arxiv      1081
10.6084/m             17
10.3929/ethz-b-       16
10.17863/cam          13
10.5281/zenodo        12
10.1021/ja            10
10.13140/rg            8
10.1007/jhep           6
10.13016/m             6
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
hdl.handle.net                 12479
dspace.mit.edu                   117
mit.edu                           29
globalchange.mit.edu              20
orcid.org                          7
None                               3
jstor.org                          1
aclanthology.org                   1
arxiv.org                          1
convention2.allacademic.com    

# E-LIS Repository

In [49]:
E_LIS_df, E_LIS_summary = get_server_data("E-LIS_Repository")


 SERVER ANALYSIS: E-LIS_REPOSITORY
  > Files found:    1
  > Raw records:    9128
  > Cleaned shape:  (9128, 91)
  > Unique DOIs:    264
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>              8864
10.13140/rg         75
10.5281/zenodo      54
10.13140/2          16
10.11575/prism      16
10.26268/heal       13
10.48550/arxiv      10
10.6084/m            9
10.4403/jlis         7
10.35050/jipm        5
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
None                   8116
eprints.rclis.org       759
doi.org                 232
dialnet.unirioja.es       3
icono14.net               2
elar.urfu.ru              2
jipm.irandoc.ac.ir        2
openaccess.uoc.edu        1
rmlconsultores.com        1
scientificia.com          1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
None       

# Earth and Space Science Open Archive

In [50]:
Earth_df, Earth_summary = get_server_data("Earth_and_Space_Science_Open_Archive")


 SERVER ANALYSIS: EARTH_AND_SPACE_SCIENCE_OPEN_ARCHIVE
  > Files found:    1
  > Raw records:    22932
  > Cleaned shape:  (22932, 91)
  > Unique DOIs:    22932
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.1002/essoar     13220
10.22541/essoar     9712
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
essopenarchive.org    22557
authorea.com            321
essoar.org               49
techrxiv.org              5
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.1002     13220
10.22541     9712
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
311    22932
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Wiley    22932
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------

# EarthArXiv

In [51]:
EarthArXiv_df, EarthArXiv_summary = get_server_data("EarthArXiv")


 SERVER ANALYSIS: EARTHARXIV
  > Files found:    2
  > Raw records:    6537
  > Cleaned shape:  (6537, 91)
  > Unique DOIs:    6537
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31223/x      4691
10.31223/osf    1777
10.31227/osf      26
10.31219/osf      18
10.31234/osf       6
10.31225/osf       6
10.31220/osf       6
10.15697/fk        2
10.31235/osf       2
10.31228/osf       2
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
eartharxiv.org        6373
osf.io                 163
dev.eartharxiv.org       1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31223    6468
10.31227      26
10.31219      18
10.31234       6
10.31225       6
10.31220       6
10.15697       2
10.31235       2
10.31228       2
10.31230       1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
----------

# EasyChair preprint

In [81]:
EasyChair_df, EasyChair_summary = get_server_data("EasyChair_preprint")


 SERVER ANALYSIS: EASYCHAIR_PREPRINT
  > Files found:    1
  > Raw records:    620
  > Cleaned shape:  (620, 91)
  > Unique DOIs:    620
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.29007/g    12
10.29007/h     9
10.29007/k     9
10.29007/t     8
10.29007/c     8
10.29007/m     7
10.29007/p     7
10.29007/z     7
10.29007/v     6
10.29007/x     6
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
easychair.org    620
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.29007    620
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
11545    620
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
EasyChair    620
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------


In [82]:
EasyChair_df

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,gold_server_name,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.29007/hsh2,EasyChair preprint,crossref,10.29007/hsh2,10.29007/hsh2,https://doi.org/10.29007/hsh2,https://easychair.org/publications/preprint/1,https://easychair.org/publications/preprint/1,10.29007,11545,None,None,crossref,EasyChair,EasyChair Preprints,None,None,2516-2314,Unification with Abstraction and Theory Instan...,None,None,None,None,report-series,None,report-series,None,False,2018-01-12,None,2018-01-22,2022-04-05,None,2017-09-13,None,2017-09-13,None,2017,issued_date,None,None,None,None,None,<jats:p>This paper explores two new inference ...,This paper explores two new inference rules fo...,None,None,"Reger, Giles; Suda, Martin; Voronkov, Andrei",None,None,"[{""affiliation"": [], ""family"": ""Reger"", ""given...",None,None,None,None,None,None,None,None,2,None,None,2,0,None,None,,,False,None,,None,None,,None,NaN,None,issn,14,None,"{""DOI"": ""10.29007/hsh2"", ""ISSN"": [""2516-2314""]...",EasyChair,10.29007/hsh2,10.29007,10.29007,hsh2,10.29007/hsh,10.29007/hsh,easychair.org,easychair.org/publications
1,crossref::10.29007/g4bq,EasyChair preprint,crossref,10.29007/g4bq,10.29007/g4bq,https://doi.org/10.29007/g4bq,https://easychair.org/publications/preprint/WjKW,https://easychair.org/publications/preprint/WjKW,10.29007,11545,None,None,crossref,EasyChair,EasyChair Preprints,None,None,2516-2314,"Reconstructing Turing's ""paper machine""",None,None,None,None,report-series,None,report-series,None,False,2018-01-12,None,2018-01-22,2022-04-04,None,2017-09-14,None,2017-09-14,None,2017,issued_date,None,None,None,None,None,<jats:p>It is an amazing fact that the very fi...,It is an amazing fact that the very first ches...,None,None,"Kasparov, Garry; Friedel, Frederic",None,None,"[{""affiliation"": [], ""family"": ""Kasparov"", ""gi...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,issn,14,None,"{""DOI"": ""10.29007/g4bq"", ""ISSN"": [""2516-2314""]...",EasyChair,10.29007/g4bq,10.29007,10.29007,g4bq,10.29007/g,10.29007/g,easychair.org,easychair.org/publications
2,crossref::10.29007/pjn4,EasyChair preprint,crossref,10.29007/pjn4,10.29007/pjn4,https://doi.org/10.29007/pjn4,https://easychair.org/publications/preprint/N2sl,https://easychair.org/publications/preprint/N2sl,10.29007,11545,None,None,crossref,EasyChair,EasyChair Preprints,None,None,2516-2314,Computation of Some Integer Sequences in Maple,None,None,None,None,report-series,None,report-series,None,False,2018-01-12,None,2018-01-22,2024-07-11,None,2017-11-20,None,2017-11-20,None,2017,issued_date,None,None,None,None,None,<jats:p>We consider some integer sequences con...,We consider some integer sequences connected w...,N

# EcoEvoRxiv

In [53]:
EcoEvoRxiv_df, EcoEvoRxiv_summary = get_server_data("EcoEvoRxiv")


 SERVER ANALYSIS: ECOEVORXIV
  > Files found:    2
  > Raw records:    2886
  > Cleaned shape:  (2886, 91)
  > Unique DOIs:    2886
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.32942/x      1948
10.32942/osf     935
10.31219/osf       3
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
ecoevorxiv.org    2838
osf.io              48
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.32942    2883
10.31219       3
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
29705    2862
15934      24
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
California Digital Library (CDL)    2862
Center for Open Science               24
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
--------------------

# EconStor Preprints

In [54]:
EconStor_df, EconStor_summary = get_server_data("EconStor_Preprints")

/tmp/ipykernel_53666/1196196344.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  raw_df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)



 SERVER ANALYSIS: ECONSTOR_PREPRINTS
  > Files found:    3
  > Raw records:    71761
  > Cleaned shape:  (71761, 91)
  > Unique DOIs:    6867
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>                     64686
10.18723/diw               588
10.13140/rg                465
10.2373/18                 429
10.1007/s                  377
10.5282/ubm                376
10.34989/swp-              292
10.24406/publica-fhg-      262
10.3929/ethz-a-            242
10.17192/es                232
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
hdl.handle.net           71010
ideas.repec.org            371
econpapers.repec.org       176
econstor.eu                 71
budrich-journals.de         55
zenodo.org                  11
journal-alm.org              7
eprints.lincoln.ac.uk        4
bnarchives.yorku.ca          4
frankfurter-hefte

# ECSarXiv

In [55]:
ECSarXiv_df, ECSarXiv_summary = get_server_data("ECSarXiv")


 SERVER ANALYSIS: ECSARXIV
  > Files found:    1
  > Raw records:    314
  > Cleaned shape:  (314, 91)
  > Unique DOIs:    314
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.1149/osf    314
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    314
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.1149    314
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
77    314
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
The Electrochemical Society    314
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    314
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None    314
Name

# EdArXiv

In [56]:
EdArXiv_df, EdArXiv_summary = get_server_data("EdArXiv")


 SERVER ANALYSIS: EDARXIV
  > Files found:    1
  > Raw records:    2547
  > Cleaned shape:  (2547, 91)
  > Unique DOIs:    2547
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.35542/osf    2547
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    2547
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.35542    2547
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    2547
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Open Science    2547
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    2547
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None   

# EGUsphere

In [85]:
EGUsphere_df, EGUsphere_summary = get_server_data("EGUsphere")


 SERVER ANALYSIS: EGUSPHERE
  > Files found:    1
  > Raw records:    15253
  > Cleaned shape:  (15253, 91)
  > Unique DOIs:    15253
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.5194/egusphere-    15249
10.5194/amt-              3
10.5194/hess-             1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
egusphere.copernicus.org          15252
oscar-egusphere.copernicus.org        1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.5194    15253
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
3145    15253
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Copernicus GmbH    15253
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None

In [87]:
pattern = r'10.5194/amt-|10.5194/hess-'
# pattern = r'v\d+$'
mask = EGUsphere_df['doi'].str.contains(pattern, regex=True, na=False)
result = EGUsphere_df[mask]
result

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,gold_server_name,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
263,crossref::10.5194/amt-2022-295,EGUsphere,crossref,10.5194/amt-2022-295,10.5194/amt-2022-295,https://doi.org/10.5194/amt-2022-295,https://egusphere.copernicus.org/preprints/202...,https://egusphere.copernicus.org/preprints/202...,10.5194,3145,None,None,crossref,Copernicus GmbH,None,None,Gases/In Situ Measurement/Instruments and Plat...,None,Temperature dependent sensitivity of iodide ch...,None,None,None,None,posted-content,preprint,preprint,None,True,2022-05-11,2022-05-11,2023-03-21,2026-02-28,None,2022-05-11,None,2022-05-11,None,2022,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>Abstract. Iodide chemical ionization m...,Abstract. Iodide chemical ionization mass spec...,None,None,"Robinson, Michael A.; Neuman, J. Andrew; Huey,...",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-0977-9...",None,None,"[{""DOI"": ""10.13039/100004800"", ""award"": [""20RD...",California Air Resources Board; NOAA Center fo...,2.0,None,None,None,1,None,None,1,0,None,"{""has-comment"": [{""asserted-by"": ""subject"", ""i...",,10.5194/amt-15-4295-2022,True,None,,None,None,10.5194/amt-2022-295-rc1;10.5194/amt-2022-295-rc2,None,NaN,None,prefix/primary_domain,2,None,"{""DOI"": ""10.5194/amt-2022-295"", ""URL"": ""https:...",Gases/In Situ Measurement/Instruments and Plat...,10.5194/amt-2022-295,10.5194,10.5194,amt-2022-295,10.5194/amt-,10.5194/amt-,egusphere.copernicus.org,egusphere.copernicus.org/preprints
264,crossref::10.5194/amt-2022-295-supplement,EGUsphere,crossref,10.5194/amt-2022-295-supplement,10.5194/amt-2022-295-supplement,https://doi.org/10.5194/amt-2022-295-supplement,https://egusphere.copernicus.org/preprints/202...,https://egusphere.copernicus.org/preprints/202...,10.5194,3145,None,None,crossref,Copernicus GmbH,None,None,Gases/In Situ Measurement/Instruments and Plat...,None,"Supplementary material to ""Temperature depende...",None,None,None,None,posted-content,other,other,None,True,2022-05-11,2022-05-11,2023-03-21,2026-02-28,None,2022-05-11,None,2022-05-11,None,2022,issued_date,posted_date,None,None,None,None,None,None,None,None,"Robinson, Michael A.; Neuman, J. Andrew; Huey,...",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-0977-9...",None,None,None,None,NaN,None,None,None,1,None,None,1,0,None,"{""is-supplement-to"": [{""asserted-by"": ""subject...",,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,2,None,"{""DOI"": ""10.5194/amt-2022-295-supplement"", ""UR...",Gases/In Situ Measurement/Instruments and Plat...,10.5194/amt-2022-295-supplement,10.5194,10.5194,amt-2022-295-supplement,10.5194/amt-,10.5194/amt-,egusphere.copernicus.org,eg

https://doi.org/10.5194/amt-2024-3967 it seems like when publish, the doi change to have the same patterns as the journal doi

# Electron Colloquium Comput Complex

In [58]:
Electron_df, Electron_summary = get_server_data("Electron_Colloquium_Comput_Complex")


 SERVER ANALYSIS: ELECTRON_COLLOQUIUM_COMPUT_COMPLEX
  > Files found:    1
  > Raw records:    227
  > Cleaned shape:  (227, 91)
  > Unique DOIs:    0
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>    227
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
dblp.uni-trier.de      221
eccc.weizmann.ac.il      6
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
None    227
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    227
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
None    227
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    227
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------


# eLife

In [89]:
elife_df, elife_summary = get_server_data("eLife")


 SERVER ANALYSIS: ELIFE
  > Files found:    2
  > Raw records:    29901
  > Cleaned shape:  (29901, 91)
  > Unique DOIs:    29901
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.7554/elife    29901
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
elifesciences.org    29901
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.7554    29901
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
4374    29901
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
eLife Sciences Publications, Ltd    29901
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
eLife    21412
None      8489
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
-------------

In [91]:
elife_df[elife_df['type_backend_raw']=='journal-article']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,gold_server_name,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.7554/elife.02516,eLife,crossref,10.7554/elife.02516,10.7554/elife.02516,https://doi.org/10.7554/elife.02516,http://elifesciences.org/lookup/doi/10.7554/eL...,http://elifesciences.org/lookup/doi/10.7554/eL...,10.7554,4374,None,None,crossref,"eLife Sciences Publications, Ltd",eLife,None,None,2050-084X,Correction: A diversity of localized timescale...,None,None,None,en,journal-article,None,journal-article,None,False,2014-08-22,None,2014-08-22,2026-03-17,None,2014-02-18,None,2014-02-18,2014-02-18,2014.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/3.0/,http://creativecommons.org/licenses/by/3.0/,None,None,None,None,"Chaudhuri, Rishidev; Bernacchia, Alberto; Wang...",None,None,"[{""affiliation"": [], ""family"": ""Chaudhuri"", ""g...",None,None,None,None,NaN,None,None,None,18,None,None,18,0,None,None,,,False,None,,None,None,,None,None,None,issn,0,None,"{""DOI"": ""10.7554/elife.02516"", ""ISSN"": [""2050-...","eLife Sciences Publications, Ltd",10.7554/elife.02516,10.7554,10.7554,elife.02516,10.7554/elife,10.7554/elife,elifesciences.org,elifesciences.org/lookup
1,crossref::10.7554/elife.08172,eLife,crossref,10.7554/elife.08172,10.7554/elife.08172,https://doi.org/10.7554/elife.08172,http://elifesciences.org/content/4/e08127,http://elifesciences.org/content/4/e08127,10.7554,4374,None,None,crossref,"eLife Sciences Publications, Ltd",eLife,None,None,2050-084X,The number of olfactory stimuli that humans ca...,None,None,None,None,journal-article,None,journal-article,None,False,2015-07-07,None,2015-07-07,2022-04-05,None,2015-07-07,None,2015-07-07,2015-07-07,2015.0,issued_date,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,None,None,issn,0,None,"{""DOI"": ""10.7554/elife.08172"", ""ISSN"": [""2050-...","eLife Sciences Publications, Ltd",10.7554/elife.08172,10.7554,10.7554,elife.08172,10.7554/elife,10.7554/elife,elifesciences.org,elifesciences.org/content
3,crossref::10.7554/elife.43558,eLife,crossref,10.7554/elife.43558,10.7554/elife.43558,https://doi.org/10.7554/elife.43558,https://elifesciences.org/articles/43558,https://elifesciences.org/articles/43558,10.7554,4374,None,None,crossref,"eLife Sciences Publications, Ltd",eLife,None,None,2050-084X,Rapid task-dependent tuning of the mouse olfac...,None,None,None,en,journal-article,None,journal-article,None,False,2019-02-06,None,2019-02-06,2026-04-14,None,2019-02-06,None,2019-02-06,2019-02-06,2019.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<jats:p>Adapting neural representation to ra

# ELPUB (Universitat Wuppertal)

In [60]:
ELPUB_df, ELPUB_summary = get_server_data("ELPUB_(Universitat_Wuppertal)")


 SERVER ANALYSIS: ELPUB_(UNIVERSITAT_WUPPERTAL)
  > Files found:    2
  > Raw records:    41
  > Cleaned shape:  (41, 91)
  > Unique DOIs:    41
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.25926/x          2
10.25926/7qdf       1
10.25926/wrn        1
10.25926/xdz        1
10.25926/cfrc-ad    1
10.25926/fjtg-wr    1
10.25926/fjdf-ae    1
10.25926/790c       1
10.25926/9bys       1
10.25926/2bhg       1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
elpub.bib.uni-wuppertal.de    41
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.25926    41
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    41
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
University of Wuppertal            39
Berg

/tmp/ipykernel_53666/1196196344.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  raw_df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)


# EmeRI

In [61]:
EmeRI_df, EmeRI_summary = get_server_data("EmeRI")


 SERVER ANALYSIS: EMERI
  > Files found:    1
  > Raw records:    8
  > Cleaned shape:  (8, 91)
  > Unique DOIs:    8
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.21452/15    6
10.21452/23    2
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprints.ibict.br    8
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.21452    8
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
8875    8
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
ABEC Publicacoes    8
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    8
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
EMERI    8
Name:

# Encyclopedia

In [62]:
Encyclopedia_df, Encyclopedia_summary = get_server_data("Encyclopedia")


 SERVER ANALYSIS: ENCYCLOPEDIA
  > Files found:    1
  > Raw records:    166
  > Cleaned shape:  (166, 91)
  > Unique DOIs:    166
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.32545/encyclopedia    166
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
encyclopedia.pub    166
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.32545    166
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
1968    166
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
MDPI AG    166
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    166
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None    1

# EnerarXiv

In [63]:
EnerarXiv_df, EnerarXiv_summary = get_server_data("EnerarXiv")


 SERVER ANALYSIS: ENERARXIV
  > Files found:    1
  > Raw records:    204
  > Cleaned shape:  (204, 91)
  > Unique DOIs:    204
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.46855/20    204
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
enerarxiv.org    204
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.46855    204
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
26239    204
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Applied Energy Innovation Institute (AEii)    204
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    204
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
instit

# engrXiv

In [64]:
engrXiv_df, engrXiv_summary = get_server_data("engrXiv")


 SERVER ANALYSIS: ENGRXIV
  > Files found:    2
  > Raw records:    4929
  > Cleaned shape:  (4929, 91)
  > Unique DOIs:    4929
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31224/osf    1909
10.31224/43       85
10.31224/30       82
10.31224/36       81
10.31224/40       80
10.31224/34       80
10.31224/42       79
10.31224/31       78
10.31224/25       78
10.31224/38       77
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
engrxiv.org    4839
osf.io           90
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31224    4642
10.31219      73
10.31234      68
10.31227      53
10.31235      39
10.31228      26
10.31223       9
10.31230       6
10.31220       5
10.31225       4
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
33966    4621

# F1000Research

In [5]:
F1000Research_df, F1000Research_summary = get_server_data("F1000Research")


 SERVER ANALYSIS: F1000RESEARCH
  > Files found:    1
  > Raw records:    16873
  > Cleaned shape:  (16873, 91)
  > Unique DOIs:    16873
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12688/f      16859
10.3410/f           7
<NA>                4
10.3410/10          1
10.3410/12          1
10.3410/http        1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
f1000research.com        16867
                             2
someurl.com                  2
xy.net                       1
researchdev.f1000.com        1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12688    16863
10.3410        10
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2560    16863
4950       10
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------

# FocUS Archive

In [6]:
FocUS_df, FocUS_summary = get_server_data("FocUS_Archive")


 SERVER ANALYSIS: FOCUS_ARCHIVE
  > Files found:    1
  > Raw records:    83
  > Cleaned shape:  (83, 91)
  > Unique DOIs:    83
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31225/osf    70
10.31227/osf     9
10.31219/osf     3
10.31234/osf     1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    83
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31225    70
10.31227     9
10.31219     3
10.31234     1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    83
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Open Science    83
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    83
Name: count, dtype: int64



# Frenxiv

In [7]:
Frenxiv_df, Frenxiv_summary = get_server_data("Frenxiv")


 SERVER ANALYSIS: FRENXIV
  > Files found:    1
  > Raw records:    179
  > Cleaned shape:  (179, 91)
  > Unique DOIs:    179
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31226/osf    178
10.31227/osf      1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    179
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31226    178
10.31227      1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    179
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Open Science    179
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    179
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
----------------------------

# Gates Open Research

In [8]:
Gates_df, Gates_summary = get_server_data("Gates_Open_Research")


 SERVER ANALYSIS: GATES_OPEN_RESEARCH
  > Files found:    1
  > Raw records:    863
  > Cleaned shape:  (863, 91)
  > Unique DOIs:    863
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12688/gatesopenres    863
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
gatesopenresearch.org    863
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12688    863
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2560    863
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
F1000 Research Ltd    863
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
Gates Open Research    862
None                     1
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTI

# HAL

In [9]:
# HAL_df, HAL_summary = get_server_data("HAL")

# HANS Publication PrePrints

In [10]:
HANS_df, HANS_summary = get_server_data("HANS_Publication_PrePrints")


 SERVER ANALYSIS: HANS_PUBLICATION_PREPRINTS
  > Files found:    1
  > Raw records:    75
  > Cleaned shape:  (75, 91)
  > Unique DOIs:    75
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12677/hanspreprints    75
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
hanspub.org    75
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12677    75
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
4945    75
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Hans Publishers    75
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
HANS Publication PrePrints    75
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
--------------------------

# HRB Open Research

In [11]:
HRB_df, HRB_summary = get_server_data("HRB_Open_Research")


 SERVER ANALYSIS: HRB_OPEN_RESEARCH
  > Files found:    1
  > Raw records:    1012
  > Cleaned shape:  (1012, 91)
  > Unique DOIs:    1012
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12688/hrbopenres    1012
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
hrbopenresearch.org    1012
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12688    1012
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2560    1012
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
F1000 Research Ltd    1012
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
HRB Open Research    1012
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------

# Humanities Commons CORE

In [12]:
CORE_df, CORE_summary = get_server_data("Humanities_Commons_CORE")


 SERVER ANALYSIS: HUMANITIES_COMMONS_CORE
  > Files found:    2
  > Raw records:    29584
  > Cleaned shape:  (29584, 91)
  > Unique DOIs:    29584
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.17613/m    2950
10.17613/j     289
10.17613/s     279
10.17613/e     273
10.17613/b     272
10.17613/r     270
10.17613/z     270
10.17613/y     270
10.17613/w     267
10.17613/t     266
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
works.hcommons.org    20522
hcommons.org           9062
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.17613    29584
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    29584
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
unknown                                 

# IACR Cryptology ePrint Archive

In [13]:
IACR_df, IACR_summary = get_server_data("IACR_Cryptology_ePrint_Archive")


 SERVER ANALYSIS: IACR_CRYPTOLOGY_EPRINT_ARCHIVE
  > Files found:    1
  > Raw records:    11904
  > Cleaned shape:  (11904, 91)
  > Unique DOIs:    208
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>                  11696
10.48550/arxiv           72
10.13140/rg              28
10.60882/cispa           25
10.5281/zenodo           18
10.1007/97               11
10.3929/ethz-b-           5
10.7916/d                 3
10.4230/dagsemproc        3
10.3217/jucs-             3
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
eprint.iacr.org         8228
dblp.uni-trier.de       3493
csrc.nist.gov             15
cacr.uwaterloo.ca         12
researchgate.net           7
caislab.kaist.ac.kr        6
131002.net                 6
cr.yp.to                   6
apps.dtic.mil              5
people.csail.mit.edu       5
Name: count, dtype: int64

# INA-Rxiv

In [14]:
INA_df, INA_summary = get_server_data("INA-Rxiv")


 SERVER ANALYSIS: INA-RXIV
  > Files found:    1
  > Raw records:    17837
  > Cleaned shape:  (17837, 91)
  > Unique DOIs:    17837
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31227/osf    15764
10.31219/osf      784
10.31234/osf      448
10.31228/osf      227
10.31235/osf      219
10.31223/osf      183
10.31230/osf       60
10.31220/osf       55
10.31229/osf       48
10.31225/osf       44
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    17837
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31227    15764
10.31219      784
10.31234      448
10.31228      227
10.31235      219
10.31223      183
10.31230       60
10.31220       55
10.31229       48
10.31225       44
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    1759

# IndiaRxiv

In [15]:
IndiaRxiv_df, IndiaRxiv_summary = get_server_data("IndiaRxiv")


 SERVER ANALYSIS: INDIARXIV
  > Files found:    2
  > Raw records:    142
  > Cleaned shape:  (142, 91)
  > Unique DOIs:    142
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.35543/osf          135
10.35543/indiarxiv      7
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io             135
ops.iihr.res.in      7
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.35543    142
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
34961    98
15934    44
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Open Access India          98
Center for Open Science    44
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None                            

# JMIR Preprints

In [16]:
JMIR_df, JMIR_summary = get_server_data("JMIR_Preprints")


 SERVER ANALYSIS: JMIR_PREPRINTS
  > Files found:    1
  > Raw records:    37631
  > Cleaned shape:  (37631, 91)
  > Unique DOIs:    37631
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.2196/preprints    37630
10.2196/iproc            1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprints.jmir.org    37631
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.2196    37631
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
1010    37631
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
JMIR Publications Inc.    37631
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    37631
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAM

# Jxiv

In [17]:
Jxiv_df, Jxiv_summary = get_server_data("Jxiv")


 SERVER ANALYSIS: JXIV
  > Files found:    1
  > Raw records:    902
  > Cleaned shape:  (902, 91)
  > Unique DOIs:    902
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.51094/jxiv    902
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
jxiv.jst.go.jp    902
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.51094    902
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    902
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Jxiv, JST Preprint Server    902
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    902
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None    9

# Keldysh Institute Preprints

In [18]:
Keldysh_df, Keldysh_summary = get_server_data("Keldysh_Institute_Preprints")


 SERVER ANALYSIS: KELDYSH_INSTITUTE_PREPRINTS
  > Files found:    1
  > Raw records:    1258
  > Cleaned shape:  (1258, 91)
  > Unique DOIs:    1258
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.20948/prepr-       1257
10.20948/preprints       1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
keldysh.ru            1257
library.keldysh.ru       1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.20948    1258
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
8521    1258
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Keldysh Institute of Applied Mathematics    1258
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
Keldysh Institute Prep

# LatArXiv

In [19]:
LatArXiv_df, LatArXiv_summary = get_server_data("LatArXiv")


 SERVER ANALYSIS: LATARXIV
  > Files found:    1
  > Raw records:    125
  > Cleaned shape:  (125, 91)
  > Unique DOIs:    125
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.62059/latarxiv    125
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprints.latarxiv.org    125
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.62059    125
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
48409    125
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Paideia Studio (publications)    125
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    125
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
insti

# Law Archive

In [20]:
Law_Archive_df, Law_Archive_summary = get_server_data("Law_Archive")


 SERVER ANALYSIS: LAW_ARCHIVE
  > Files found:    1
  > Raw records:    1808
  > Cleaned shape:  (1808, 91)
  > Unique DOIs:    1808
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31228/osf    837
10.31219/osf    470
10.31227/osf    241
10.31234/osf    175
10.31235/osf     38
10.31231/osf     17
10.31223/osf     12
10.31224/osf      6
10.31230/osf      6
10.31229/osf      3
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    1808
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31228    837
10.31219    470
10.31227    241
10.31234    175
10.31235     38
10.31231     17
10.31223     12
10.31224      6
10.31230      6
10.31229      3
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    1794
29705      12
33966       2
Name: count,

# LIS Scholarship Archive

In [21]:
LIS_df, LIS_summary = get_server_data("LIS_Scholarship_Archive")


 SERVER ANALYSIS: LIS_SCHOLARSHIP_ARCHIVE
  > Files found:    1
  > Raw records:    397
  > Cleaned shape:  (397, 91)
  > Unique DOIs:    397
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31229/osf    290
10.31219/osf     44
10.31227/osf     21
10.31228/osf      9
10.31235/osf      7
10.31234/osf      7
10.31223/osf      7
10.31230/osf      5
10.31225/osf      3
10.31220/osf      2
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    397
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31229    290
10.31219     44
10.31227     21
10.31228      9
10.31235      7
10.31234      7
10.31223      7
10.31230      5
10.31225      3
10.31220      2
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    388
29705      7
242        2
Name: c

# LSE Research Online Documents on Economics

In [22]:
LSE_df, LSE_summary = get_server_data("LSE_Research_Online_Documents_on_Economics")


 SERVER ANALYSIS: LSE_RESEARCH_ONLINE_DOCUMENTS_ON_ECONOMICS
  > Files found:    1
  > Raw records:    119
  > Cleaned shape:  (119, 91)
  > Unique DOIs:    12
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>           107
10.5089/97       3
10.1429/85       2
10.7208/97       1
10.2760/27       1
10.1425/84       1
10.60692/b       1
10.13140/rg      1
10.7916/d        1
10.1108/s        1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
ideas.repec.org          74
eprints.lse.ac.uk        24
cep.lse.ac.uk             5
sticerd.lse.ac.uk         3
elibrary.imf.org          2
foundation.org.uk         1
cris.unibo.it             1
europepmc.org             1
carnegieendowment.org     1
dialnet.unirioja.es       1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
None        107
10.5089     

# MarXiv

In [23]:
MarXiv_df, MarXiv_summary = get_server_data("MarXiv")


 SERVER ANALYSIS: MARXIV
  > Files found:    1
  > Raw records:    508
  > Cleaned shape:  (508, 91)
  > Unique DOIs:    508
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31230/osf    317
10.31227/osf     60
10.31219/osf     57
10.31235/osf     23
10.31228/osf     19
10.31234/osf     13
10.31223/osf      9
10.31220/osf      5
10.31225/osf      5
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    508
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31230    317
10.31227     60
10.31219     57
10.31235     23
10.31228     19
10.31234     13
10.31223      9
10.31220      5
10.31225      5
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    494
29705      9
242        5
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
-----

# MediArXiv

In [24]:
MediArXiv_df, MediArXiv_summary = get_server_data("MediArXiv")


 SERVER ANALYSIS: MEDIARXIV
  > Files found:    1
  > Raw records:    309
  > Cleaned shape:  (309, 91)
  > Unique DOIs:    309
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.33767/osf    306
10.31219/osf      3
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    309
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.33767    306
10.31219      3
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    309
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Open Science    309
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    309
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
--------------------------

# medRxiv

In [25]:
medRxiv_df, medRxiv_summary = get_server_data("medRxiv")


 SERVER ANALYSIS: MEDRXIV
  > Files found:    1
  > Raw records:    75743
  > Cleaned shape:  (75743, 91)
  > Unique DOIs:    75743
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.1101/20    74951
10.1101/19      792
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
medrxiv.org    75743
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.1101    75743
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
246    75743
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Cold Spring Harbor Laboratory    75743
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    75743
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
--------------------

# MetaArXiv

In [26]:
MetaArXiv_df, MetaArXiv_summary = get_server_data("MetaArXiv")


 SERVER ANALYSIS: METAARXIV
  > Files found:    1
  > Raw records:    880
  > Cleaned shape:  (880, 91)
  > Unique DOIs:    880
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31222/osf    851
10.31219/osf     11
10.31227/osf      7
10.31234/osf      4
10.31235/osf      4
10.31231/osf      1
10.31223/osf      1
10.31228/osf      1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    880
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31222    851
10.31219     11
10.31227      7
10.31234      4
10.31235      4
10.31231      1
10.31223      1
10.31228      1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    879
29705      1
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for

# MindRxiv

In [27]:
MindRxiv_df, MindRxiv_summary = get_server_data("MindRxiv")


 SERVER ANALYSIS: MINDRXIV
  > Files found:    1
  > Raw records:    335
  > Cleaned shape:  (335, 91)
  > Unique DOIs:    335
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31231/osf    258
10.31227/osf     38
10.31219/osf     15
10.31234/osf      7
10.31228/osf      5
10.31235/osf      5
10.31229/osf      4
10.31223/osf      2
10.31225/osf      1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    335
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31231    258
10.31227     38
10.31219     15
10.31234      7
10.31228      5
10.31235      5
10.31229      4
10.31223      2
10.31225      1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    333
29705      2
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
----------------

# MNI Open Research

In [28]:
MNI_df, MNI_summary = get_server_data("MNI_Open_Research")


 SERVER ANALYSIS: MNI_OPEN_RESEARCH
  > Files found:    1
  > Raw records:    20
  > Cleaned shape:  (20, 91)
  > Unique DOIs:    20
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12688/mniopenres    20
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
mniopenresearch.org    20
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12688    20
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2560    20
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
F1000 Research Ltd    20
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
MNI Open Research    19
None                  1
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------

# Munich Personal RePEc Archive

In [29]:
Munich_df, Munich_summary = get_server_data("Munich_Personal_RePEc_Archive")

/tmp/ipykernel_2658/1196196344.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  raw_df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)



 SERVER ANALYSIS: MUNICH_PERSONAL_REPEC_ARCHIVE
  > Files found:    7
  > Raw records:    68692
  > Cleaned shape:  (68692, 91)
  > Unique DOIs:    1427
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>              67264
10.13140/rg         476
10.48550/arxiv      333
10.5281/zenodo      100
10.13140/2           60
10.22004/ag          25
10.6084/m            15
10.17605/osf         12
10.17192/es           7
10.1108/ijse-         7
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
theses.fr                    50259
None                         16501
doi.org                        899
escholarship.org               449
arxiv.org                      191
library.tue.nl                 120
hdl.handle.net                  70
eprints.iisc.ac.in              48
eref.uni-bayreuth.de            30
repositorio.banrep.gov.co       24
Name:

# National Bureau of Economic Research

In [30]:
National_Bureau_df, National_Bureau_summary = get_server_data("National_Bureau_of_Economic_Research")


 SERVER ANALYSIS: NATIONAL_BUREAU_OF_ECONOMIC_RESEARCH
  > Files found:    5
  > Raw records:    1856
  > Cleaned shape:  (1856, 91)
  > Unique DOIs:    28
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>                      1828
10.7916/d                    9
10.7208/97                   5
10.5089/97                   3
10.1007/s                    3
10.17863/cam                 2
10.1002/10                   1
10.23668/psycharchives       1
10.13140/2                   1
10.57912/23                  1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
ideas.repec.org                  1337
eric.ed.gov                       485
econpapers.repec.org                7
ecsocman.hse.ru                     4
nber.org                            3
healthpolicy.fsi.stanford.edu       2
elibrary.imf.org                    2
europepmc.org   

# Nature Precedings

In [31]:
Nature_df, Nature_summary = get_server_data("Nature_Precedings")


 SERVER ANALYSIS: NATURE_PRECEDINGS
  > Files found:    1
  > Raw records:    5210
  > Cleaned shape:  (5210, 91)
  > Unique DOIs:    5210
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.1038/npre    5210
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
nature.com               3312
precedings.nature.com    1898
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.1038    5210
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
297    5210
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Springer Science and Business Media LLC    5210
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
Nature Precedings    5172
None                   38
Name: coun

# NewAddictionsX

In [32]:
NewAddictionsX_df, NewAddictionsX_summary = get_server_data("NewAddictionsX")


 SERVER ANALYSIS: NEWADDICTIONSX
  > Files found:    1
  > Raw records:    7
  > Cleaned shape:  (7, 91)
  > Unique DOIs:    7
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31219/osf    7
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    7
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31219    7
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    7
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Open Science    7
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    7
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None    7
Name: count, dtyp

# NutriXiv

In [33]:
NutriXiv_df, NutriXiv_summary = get_server_data("NutriXiv")


 SERVER ANALYSIS: NUTRIXIV
  > Files found:    1
  > Raw records:    94
  > Cleaned shape:  (94, 91)
  > Unique DOIs:    94
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31232/osf    70
10.31219/osf    12
10.31228/osf     6
10.31227/osf     5
10.31235/osf     1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    94
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31232    70
10.31219    12
10.31228     6
10.31227     5
10.31235     1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    94
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Open Science    94
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    9

# Open Research Africa

In [34]:
openra_df, openra_summary = get_server_data("Open_Research_Africa")


 SERVER ANALYSIS: OPEN_RESEARCH_AFRICA
  > Files found:    1
  > Raw records:    288
  > Cleaned shape:  (288, 91)
  > Unique DOIs:    288
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12688/aasopenres       222
10.12688/openresafrica     66
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
aasopenresearch.org       176
openresearchafrica.org    112
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12688    288
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2560    288
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
F1000 Research Ltd    288
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
AAS Open Research       175
Open Research Afr

# Open Research Europe

In [35]:
openre_df, openre_summary = get_server_data("Open_Research_Europe")


 SERVER ANALYSIS: OPEN_RESEARCH_EUROPE
  > Files found:    1
  > Raw records:    1877
  > Cleaned shape:  (1877, 91)
  > Unique DOIs:    1877
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12688/openreseurope    1877
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
open-research-europe.ec.europa.eu    1877
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12688    1877
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2560    1877
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
F1000 Research Ltd    1877
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
Open Research Europe    1877
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_

# Open Science Framework

In [36]:
osf_df, osf_summary = get_server_data("Open_Science_Framework")

/tmp/ipykernel_2658/1196196344.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  raw_df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)



 SERVER ANALYSIS: OPEN_SCIENCE_FRAMEWORK
  > Files found:    3
  > Raw records:    119481
  > Cleaned shape:  (119481, 91)
  > Unique DOIs:    119481
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31219/osf    101598
10.17605/osf     15829
10.31227/osf       649
10.31234/osf       434
10.31235/osf       421
10.31228/osf       213
10.31223/osf       110
10.31220/osf        56
10.31229/osf        55
10.31224/osf        47
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io               112545
doi.org                4615
psyarxiv.com           1327
eartharxiv.org          262
thesiscommons.org       216
marxiv.org              158
engrxiv.org             137
arabixiv.org             82
mindrxiv.org             46
agrixiv.org              40
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefi

In [92]:
osf_df

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,gold_server_name,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.31219/osf.io/28kyc,Open Science Framework,crossref,10.31219/osf.io/28kyc,10.31219/osf.io/28kyc,https://doi.org/10.31219/osf.io/28kyc,https://osf.io/28kyc,https://osf.io/28kyc,10.31219,15934,None,None,crossref,Center for Open Science,None,None,Open Science Framework,None,,None,None,None,None,posted-content,preprint,preprint,None,True,2018-07-02,2016-09-27,2018-07-02,2022-04-01,None,2016-09-27,None,2016-09-27,None,2016,issued_date,posted_date,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,None,None,group_title,2,None,"{""DOI"": ""10.31219/osf.io/28kyc"", ""URL"": ""https...",Open Science Framework,10.31219/osf.io/28kyc,10.31219,10.31219,osf.io/28kyc,10.31219/osf,10.31219/osf,osf.io,osf.io/28kyc
1,crossref::10.31219/osf.io/2m2vu,Open Science Framework,crossref,10.31219/osf.io/2m2vu,10.31219/osf.io/2m2vu,https://doi.org/10.31219/osf.io/2m2vu,https://osf.io/2m2vu,https://osf.io/2m2vu,10.31219,15934,None,None,crossref,Center for Open Science,None,None,Open Science Framework,None,,None,None,None,None,posted-content,preprint,preprint,None,True,2018-07-02,2016-12-10,2018-07-02,2022-03-30,None,2016-12-10,None,2016-12-10,None,2016,issued_date,posted_date,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,None,None,group_title,2,None,"{""DOI"": ""10.31219/osf.io/2m2vu"", ""URL"": ""https...",Open Science Framework,10.31219/osf.io/2m2vu,10.31219,10.31219,osf.io/2m2vu,10.31219/osf,10.31219/osf,osf.io,osf.io/2m2vu
2,crossref::10.31219/osf.io/2twgy,Open Science Framework,crossref,10.31219/osf.io/2twgy,10.31219/osf.io/2twgy,https://doi.org/10.31219/osf.io/2twgy,https://osf.io/2twgy,https://osf.io/2twgy,10.31219,15934,None,None,crossref,Center for Open Science,None,None,Open Science Framework,None,,None,None,None,None,posted-content,preprint,preprint,None,True,2018-07-02,2016-08-29,2018-07-02,2022-03-29,None,2016-08-29,None,2016-08-29,None,2016,issued_date,posted_date,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,None,None,group_title,2,None,"{""DOI"": ""10.31219/osf.io/2twgy"", ""URL"": ""https...",Open Science Framework,10.31219/osf.io/2twgy,10.31219,10.31219,osf.io/2twgy,10.31219/osf,10.31219/osf,osf.io,osf.io/2twgy
3,crossref::10.31219/osf.io/38987,Open Science Framework,crossref,10.31219/osf.io/38987,10.31219/osf.io/38987,https://doi.org/10.31219/osf.io/38987,https://osf.io/38987,https://osf.io/38987,10.31219,15934,Non

# Organic Eprints

In [37]:
Organic_Eprints_df, Organic_Eprints_summary = get_server_data("Organic_Eprints")


 SERVER ANALYSIS: ORGANIC_EPRINTS
  > Files found:    1
  > Raw records:    13983
  > Cleaned shape:  (13983, 91)
  > Unique DOIs:    252
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>              13704
10.5281/zenodo       35
10.3220/rep          30
10.13140/rg          29
10.15454/1           22
10.5169/seals-       14
10.22004/ag          13
10.13140/2           10
10.3920/97            4
10.5073/jfk           3
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
None             12257
orgprints.org     1441
doi.org            260
                    24
bioaktuell.ch        1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
None        13704
10.17180       71
10.13140       39
10.15454       36
10.5281        35
10.3220        32
10.5169        14
10.22004       13
10.3920         4
10.

# Oroboros Instruments

In [38]:
Oroboros_Instruments_df, Oroboros_Instruments_summary = get_server_data("Oroboros_Instruments")


 SERVER ANALYSIS: OROBOROS_INSTRUMENTS
  > Files found:    2
  > Raw records:    95
  > Cleaned shape:  (95, 91)
  > Unique DOIs:    95
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.26124/mitofit    67
10.26124/bec        19
10.26124/becprep     9
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
wiki.oroboros.at                    37
bioenergetics-communications.org    28
mitofit.org                         27
mitoeagle.org                        3
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.26124    95
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    95
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
MitoFit Preprints                49
MitoFit Preprint Archives        18
Bioener

/tmp/ipykernel_2658/1196196344.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  raw_df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)


# PaleorXiv

In [39]:
PaleorXiv_df, PaleorXiv_summary = get_server_data("PaleorXiv")


 SERVER ANALYSIS: PALEORXIV
  > Files found:    1
  > Raw records:    287
  > Cleaned shape:  (287, 91)
  > Unique DOIs:    287
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31233/osf    202
10.31227/osf     23
10.31219/osf     18
10.31231/osf     17
10.31235/osf      8
10.31228/osf      5
10.31229/osf      5
10.31234/osf      3
10.31223/osf      3
10.31230/osf      2
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    287
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31233    202
10.31227     23
10.31219     18
10.31231     17
10.31235      8
10.31228      5
10.31229      5
10.31234      3
10.31223      3
10.31230      2
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    284
29705      3
Name: count, dtype: int64



TOP V

# PeerJ Preprints

In [40]:
PeerJ_df, PeerJ_summary = get_server_data("PeerJ_Preprints")


 SERVER ANALYSIS: PEERJ_PREPRINTS
  > Files found:    1
  > Raw records:    6446
  > Cleaned shape:  (6446, 91)
  > Unique DOIs:    6446
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.7287/peerj    6446
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
peerj.com    6446
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.7287    6446
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
4443    6446
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
PeerJ    6446
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    6446
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None    6446
Na

# PhilSci-Archive

In [41]:
PhilSci_df, PhilSci_summary = get_server_data("PhilSci-Archive")


 SERVER ANALYSIS: PHILSCI-ARCHIVE
  > Files found:    1
  > Raw records:    2362
  > Cleaned shape:  (2362, 91)
  > Unique DOIs:    195
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>              2167
10.48550/arxiv     146
10.13140/rg         18
10.6084/m            5
10.5281/zenodo       3
10.17863/cam         2
10.2143/lea          2
10.22381/rcp         1
10.24338/abs-        1
10.25455/wgtn        1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
None                        1676
philsci-archive.pitt.edu     505
doi.org                      181
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
None        2167
10.48550     146
10.13140      19
10.6084        5
10.5281        3
10.17863       2
10.2143        2
10.22381       1
10.25455       1
10.24338       1
Name: count, dtype: int

# PoolText

In [42]:
PoolText_df, PoolText_summary = get_server_data("PoolText")


 SERVER ANALYSIS: POOLTEXT
  > Files found:    1
  > Raw records:    79
  > Cleaned shape:  (79, 91)
  > Unique DOIs:    79
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31923/pooltext-preprint-    78
10.31923/55                     1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
content.pooltext.com    79
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31923    79
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
16838    79
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
PoolText, Inc    79
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    79
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
-------------------

# prepare@u

In [43]:
prepare_df, prepare_summary = get_server_data("prepare@u")


 SERVER ANALYSIS: PREPARE@U
  > Files found:    1
  > Raw records:    227
  > Cleaned shape:  (227, 91)
  > Unique DOIs:    227
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.36375/prepare    227
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprint.prepare.org.in    223
prepare.enggtalks.com        2
prepare.org.in               2
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.36375    227
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
22141    227
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
CALNESTOR Knowledge Solutions Private Limited    227
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None                           

# Preprints.org

In [44]:
Preprints_df, Preprints_summary = get_server_data("Preprints.org")


 SERVER ANALYSIS: PREPRINTS.ORG
  > Files found:    1
  > Raw records:    115815
  > Cleaned shape:  (115815, 91)
  > Unique DOIs:    115815
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.20944/preprints    115815
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprints.org    115815
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.20944    115815
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
1968    115815
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
MDPI AG    115815
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    115815
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
inst

# PREPRINTS.RU

In [45]:
PREPRINTS_df, PREPRINTS_summary = get_server_data("PREPRINTS.RU")


 SERVER ANALYSIS: PREPRINTS.RU
  > Files found:    1
  > Raw records:    1415
  > Cleaned shape:  (1415, 91)
  > Unique DOIs:    1415
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.24108/preprints-    1415
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprints.ru    1415
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.24108    1415
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
10196    1415
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
NPG Publishing    1415
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    1415
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_nam

# Prepublicaciones OpenCiencia

In [46]:
OpenCiencia_df, OpenCiencia_summary = get_server_data("Prepublicaciones_OpenCiencia")


 SERVER ANALYSIS: PREPUBLICACIONES_OPENCIENCIA
  > Files found:    1
  > Raw records:    8
  > Cleaned shape:  (8, 91)
  > Unique DOIs:    8
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.47073/preprints    8
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
prepublicaciones.org    8
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.47073    8
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
28319    8
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Centro de Investigacion sobre Desarrollo Humano y Sociedad (Coideso)    8
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    8
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_N

# PropylaeumDok

In [47]:
PropylaeumDok_df, PropylaeumDok_summary = get_server_data("PropylaeumDok")


 SERVER ANALYSIS: PROPYLAEUMDOK
  > Files found:    2
  > Raw records:    6750
  > Cleaned shape:  (6750, 91)
  > Unique DOIs:    6750
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.11588/propylaeumdok    6750
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
archiv.ub.uni-heidelberg.de    6750
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.11588    6750
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    6750
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Heidelberg University Library    6749
None                                1
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    6750
Name: count, dtype: int64



TOP VALU

# PsyArXiv

In [48]:
PsyArXiv_df, PsyArXiv_summary = get_server_data("PsyArXiv")


 SERVER ANALYSIS: PSYARXIV
  > Files found:    1
  > Raw records:    56866
  > Cleaned shape:  (56866, 91)
  > Unique DOIs:    56866
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31234/osf    54938
10.31219/osf      568
10.31227/osf      531
10.31235/osf      339
10.31228/osf      181
10.31223/osf       52
10.31225/osf       47
10.31230/osf       46
10.31229/osf       46
10.31224/osf       42
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    56866
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31234    54938
10.31219      568
10.31227      531
10.31235      339
10.31228      181
10.31223       52
10.31225       47
10.31230       46
10.31229       46
10.31224       42
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    5676

# Qeios

In [49]:
Qeios_df, Qeios_summary = get_server_data("Qeios")


 SERVER ANALYSIS: QEIOS
  > Files found:    1
  > Raw records:    5650
  > Cleaned shape:  (5650, 91)
  > Unique DOIs:    5650
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.32388/i    66
10.32388/g    54
10.32388/h    51
10.32388/a    49
10.32388/p    48
10.32388/j    48
10.32388/r    48
10.32388/n    48
10.32388/z    47
10.32388/o    46
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
qeios.com    5650
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.32388    5650
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
17262    5650
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Qeios Ltd    5650
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_

# RePEc: Research Papers in Economics

In [50]:
# RePEc_df, RePEc_summary = get_server_data("RePEc_Research_Papers_in_Economics")

# Research Square

In [51]:
# Research_Square_df, Research_Square_summary = get_server_data("Research_Square")

# ResearchGate

In [52]:
ResearchGate_df, ResearchGate_summary = get_server_data("ResearchGate")


 SERVER ANALYSIS: RESEARCHGATE
  > Files found:    1
  > Raw records:    181231
  > Cleaned shape:  (181231, 91)
  > Unique DOIs:    181231
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.13140/rg    181188
10.13140/2         43
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
researchgate.net    178343
rgdoi.net             2888
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.13140    181231
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    181231
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Unpublished                                                180824
Plekhanov Russian University of Economics                      21
Flora Montiberica.org                                         

# ResearchHub

In [53]:
ResearchHub_df, ResearchHub_summary = get_server_data("ResearchHub")


 SERVER ANALYSIS: RESEARCHHUB
  > Files found:    1
  > Raw records:    1636
  > Cleaned shape:  (1636, 91)
  > Unique DOIs:    1636
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.55277/researchhub    1554
10.55277/rhj              81
10.55277/pvtglk            1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
researchhub.com                1513
staging.researchhub.com         100
staging-web.researchhub.com      23
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.55277    1636
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
33940    1636
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
ResearchHub Technologies, Inc.    1636
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
-------

# SAE Mobilus®

In [54]:
SAE_df, SAE_summary = get_server_data("SAE_Mobilus®")


 SERVER ANALYSIS: SAE_MOBILUS®
  > Files found:    1
  > Raw records:    105
  > Cleaned shape:  (105, 91)
  > Unique DOIs:    105
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.47953/sae-pp-    105
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
saemobilus.sae.org    105
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.47953    105
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2796    105
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
SAE International    105
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    105
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
No

# SciELO Preprints

In [55]:
SciELO_df, SciELO_summary = get_server_data("SciELO_Preprints")


 SERVER ANALYSIS: SCIELO_PREPRINTS
  > Files found:    1
  > Raw records:    4141
  > Cleaned shape:  (4141, 91)
  > Unique DOIs:    4141
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.1590/scielopreprints        3939
10.1590/16                       87
10.1590/s                        39
10.1590/22                       29
10.1590/25                       22
10.1590/01                        9
10.1590/23                        3
10.1590/19                        3
10.1590/26                        3
10.1590/scielopreprintstest       2
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
preprints.scielo.org            4136
homolog-preprints.scielo.org       4
preprints-bolha.scielo.org         1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.1590    4141
Name: count, dtype: int64



TOP 

# ScienceOpen Preprints

In [9]:
ScienceOpen_df, ScienceOpen_summary = get_server_data("ScienceOpen_Preprints")


 SERVER ANALYSIS: SCIENCEOPEN_PREPRINTS
  > Files found:    1
  > Raw records:    2970
  > Cleaned shape:  (2970, 91)
  > Unique DOIs:    2970
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.14293/s     1386
10.14293/pr    1068
10.14293/p      516
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
scienceopen.com    2970
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.14293    2970
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
5403    2970
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
ScienceOpen    2970
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    2970
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------

In [10]:
ScienceOpen_df

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json,gold_server_name,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
0,crossref::10.14293/s2199-1006.1.sor-.ppybjel.v1,ScienceOpen Preprints,crossref,10.14293/s2199-1006.1.sor-.ppybjel.v1,10.14293/s2199-1006.1.sor-.ppybjel.v1,https://doi.org/10.14293/s2199-1006.1.sor-.ppy...,https://scienceopen.com/document?vid=e3b0997a-...,https://scienceopen.com/document?vid=e3b0997a-...,10.14293,5403,None,None,crossref,ScienceOpen,None,ScienceOpen,None,None,Towards Computationally Creating Multi-answer ...,None,None,None,None,posted-content,preprint,preprint,None,True,2020-06-14,2019-01-10,2020-09-08,2025-05-14,None,2019-01-10,None,2019-01-10,None,2019,issued_date,posted_date,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns7:p>The Remote Associates Test is a creativ...,The Remote Associates Test is a creativity tes...,"[{""URL"": ""https://scienceopen.com/document?vid...",None,"Olteteanu, Ana-Maria; Yoopoo, Kunkanit",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-0639-7...",None,None,None,None,NaN,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,4,None,"{""DOI"": ""10.14293/s2199-1006.1.sor-.ppybjel.v1...",ScienceOpen,10.14293/s2199-1006.1.sor-.ppybjel.v1,10.14293,10.14293,s2199-1006.1.sor-.ppybjel.v1,10.14293/s,10.14293/s,scienceopen.com,scienceopen.com/document
1,crossref::10.14293/s2199-1006.1.sor-.ppzm4he.v1,ScienceOpen Preprints,crossref,10.14293/s2199-1006.1.sor-.ppzm4he.v1,10.14293/s2199-1006.1.sor-.ppzm4he.v1,https://doi.org/10.14293/s2199-1006.1.sor-.ppz...,https://scienceopen.com/document?vid=e8537577-...,https://scienceopen.com/document?vid=e8537577-...,10.14293,5403,None,None,crossref,ScienceOpen,None,ScienceOpen,None,None,Towards exploring adaptive and associatively r...,None,None,None,None,posted-content,other,other,None,True,2020-08-04,2019-01-10,2020-09-08,2025-05-14,None,2019-01-10,None,2019-01-10,None,2019,issued_date,posted_date,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns7:p>This poster presents an initial set of ...,This poster presents an initial set of observa...,"[{""URL"": ""https://scienceopen.com/document?vid...",None,"Olteteanu, Ana-Maria; Dyer, Jonathan",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-0639-7...",None,None,None,None,NaN,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,4,None,"{""DOI"": ""10.14293/s2199-1006.1.sor-.ppzm4he.v1...",ScienceOpen,10.14293/s2199-1006.1.sor-.ppzm4he.v1,10.14293,10.14293,s2199-1006.1.sor-.ppzm4he.v1,10.14293/s,10.14293/s,scienceopen.com,scienceopen.com/document
2,crossref::10.14293/s2199-1006.1.sor-.ppbgbyn.

# Sciencepaper Online

In [57]:
Sciencepaper_df, Sciencepaper_summary = get_server_data("Sciencepaper_Online")


 SERVER ANALYSIS: SCIENCEPAPER_ONLINE
  > Files found:    1
  > Raw records:    99
  > Cleaned shape:  (99, 91)
  > Unique DOIs:    99
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.61951/sciencepaperonline    99
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
paper.edu.cn    99
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.61951    99
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
48263    99
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Center for Scientific Research and Development in Higher Education Institutes, Ministry of Education    99
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    99
Name: count, dtype: int64

# searchRxiv

In [58]:
searchRxiv_df, searchRxiv_summary = get_server_data("searchRxiv")


 SERVER ANALYSIS: SEARCHRXIV
  > Files found:    2
  > Raw records:    1234
  > Cleaned shape:  (1234, 91)
  > Unique DOIs:    1234
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.1079/searchrxiv    1234
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
cabidigitallibrary.org    1234
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.1079    1234
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
242    1234
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
CABI Publishing    1234
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None          1232
searchRxiv       2
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
----------------

# SocArXiv

In [59]:
SocArXiv_df, SocArXiv_summary = get_server_data("SocArXiv")


 SERVER ANALYSIS: SOCARXIV
  > Files found:    1
  > Raw records:    21541
  > Cleaned shape:  (21541, 91)
  > Unique DOIs:    21541
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31235/osf    19755
10.31219/osf      594
10.31234/osf      385
10.31227/osf      357
10.31228/osf      133
10.31223/osf      103
10.31224/osf       95
10.31231/osf       35
10.31229/osf       30
10.31225/osf       29
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    21541
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31235    19755
10.31219      594
10.31234      385
10.31227      357
10.31228      133
10.31223      103
10.31224       95
10.31231       35
10.31229       30
10.31225       29
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    2139

# Social Science Open Access Repository

In [60]:
Social_Science_Open_Access_Repository_df, Social_Science_Open_Access_Repository_summary = get_server_data("Social_Science_Open_Access_Repository")


 SERVER ANALYSIS: SOCIAL_SCIENCE_OPEN_ACCESS_REPOSITORY
  > Files found:    1
  > Raw records:    27201
  > Cleaned shape:  (27201, 91)
  > Unique DOIs:    6679
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>                      20522
10.12759/hsr               1411
10.21241/ssoar              815
10.23668/psycharchives      437
10.14765/zzf                365
10.17169/fqs-               296
10.5281/zenodo              216
10.15464/isi                202
10.14764/10                 194
10.11588/iqas               189
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
ssoar.info                     26545
wbv.de                           267
publikationen.soziologie.de      135
doi.org                          111
hdl.handle.net                    31
journals.sub.uni-hamburg.de       21
uni-graz.at                       12
verbrauc

# SportRxiv

In [61]:
SportRxiv_df, SportRxiv_summary = get_server_data("SportRxiv")


 SERVER ANALYSIS: SPORTRXIV
  > Files found:    2
  > Raw records:    878
  > Cleaned shape:  (878, 91)
  > Unique DOIs:    878
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.51224/srxiv    492
10.31236/osf      350
10.31227/osf       12
10.31234/osf        6
10.31231/osf        6
10.31219/osf        5
10.31223/osf        4
10.31235/osf        2
10.31229/osf        1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
sportrxiv.org    492
osf.io           386
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.51224    492
10.31236    350
10.31227     12
10.31234      6
10.31231      6
10.31219      5
10.31223      4
10.31235      2
10.31229      1
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
7569     492
15934    382
29705      4
Name: count

# SSRN

In [62]:
# SSRN_df, SSRN_summary = get_server_data("SSRN")

# TechRxiv

In [63]:
TechRxiv_df, TechRxiv_summary = get_server_data("TechRxiv")


 SERVER ANALYSIS: TECHRXIV
  > Files found:    1
  > Raw records:    29418
  > Cleaned shape:  (29418, 91)
  > Unique DOIs:    29418
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.36227/techrxiv    29418
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
techrxiv.org             29088
techrxiv.figshare.com      324
essopenarchive.org           5
figshare.com                 1
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.36227    29418
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
263    29418
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Institute of Electrical and Electronics Engineers (IEEE)    29418
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------

# Therapoid

In [64]:
Therapoid_df, Therapoid_summary = get_server_data("Therapoid")


 SERVER ANALYSIS: THERAPOID
  > Files found:    1
  > Raw records:    7
  > Cleaned shape:  (7, 91)
  > Unique DOIs:    7
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.24973/20    7
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
therapoid.net    7
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.24973    7
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
10597    7
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Therapoid            6
Open Therapeutics    1
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    7
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None    7

# Thesis Commons

In [65]:
Thesis_df, Thesis_summary = get_server_data("Thesis_Commons")


 SERVER ANALYSIS: THESIS_COMMONS
  > Files found:    1
  > Raw records:    3959
  > Cleaned shape:  (3959, 91)
  > Unique DOIs:    3959
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.31237/osf    3663
10.31227/osf     103
10.31231/osf      45
10.31219/osf      38
10.31228/osf      30
10.31234/osf      18
10.31230/osf      18
10.31235/osf      14
10.31223/osf      10
10.31220/osf      10
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
osf.io    3959
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.31237    3663
10.31227     103
10.31231      45
10.31219      38
10.31228      30
10.31234      18
10.31230      18
10.31235      14
10.31223      10
10.31220      10
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
15934    3939
242        10
297

# UCL Open Environment

In [66]:
UCL_df, UCL_summary = get_server_data("UCL_Open_Environment")


 SERVER ANALYSIS: UCL_OPEN_ENVIRONMENT
  > Files found:    2
  > Raw records:    369
  > Cleaned shape:  (369, 91)
  > Unique DOIs:    369
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.14324/11                317
10.14324/ucloepreprints     51
10.14324/ucloe               1
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
journals.uclpress.co.uk    364
ucl.scienceopen.com          5
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.14324    369
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
5433    369
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
UCL Press    369
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None               

/tmp/ipykernel_2658/1196196344.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  raw_df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)


# UnisaRxiv

In [67]:
UnisaRxiv_df, UnisaRxiv_summary = get_server_data("UnisaRxiv")


 SERVER ANALYSIS: UNISARXIV
  > Files found:    1
  > Raw records:    126
  > Cleaned shape:  (126, 91)
  > Unique DOIs:    126
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.25159/unisarxiv    126
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
scienceopen.com    126
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.25159    126
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
10792    126
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
UNISA Press    126
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    126
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
ScienceOpen

# VeriXiv

In [68]:
VeriXiv_df, VeriXiv_summary = get_server_data("VeriXiv")


 SERVER ANALYSIS: VERIXIV
  > Files found:    1
  > Raw records:    504
  > Cleaned shape:  (504, 91)
  > Unique DOIs:    504
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12688/verixiv    504
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
verixiv.org    504
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12688    504
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2560    504
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
F1000 Research Ltd    504
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
None    504
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAME
------------------------------
institution_name
None    504
N

# viXra

In [69]:
viXra_df, viXra_summary = get_server_data("viXra")


 SERVER ANALYSIS: VIXRA
  > Files found:    1
  > Raw records:    25570
  > Cleaned shape:  (25570, 91)
  > Unique DOIs:    2646
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
<NA>              22924
10.5281/zenodo      989
10.13140/rg         989
10.6084/m           225
10.48550/arxiv      136
10.13140/2           36
10.1016/j            23
10.17605/osf         13
10.18147/smn         11
10.1007/s            10
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
vixra.org                     25200
rxiv.org                         75
fs.gallup.unm.edu                67
philpapers.org                   25
gallup.unm.edu                   23
s3-eu-west-1.amazonaws.com       10
deepblue.lib.umich.edu           10
zenodo.org                       10
redshift.vif.com                  9
researchgate.net                  8
Name: count, dtype:

# Wellcome Open Research

In [70]:
Wellcome_df, Wellcome_summary = get_server_data("Wellcome_Open_Research")


 SERVER ANALYSIS: WELLCOME_OPEN_RESEARCH
  > Files found:    1
  > Raw records:    4727
  > Cleaned shape:  (4727, 91)
  > Unique DOIs:    4727
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.12688/wellcomeopenres    4727
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
wellcomeopenresearch.org    4727
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.12688    4727
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
2560    4727
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
F1000 Research Ltd    4727
Name: count, dtype: int64



TOP VALUES FOR: CONTAINER_TITLE
------------------------------
container_title
Wellcome Open Research    4727
Name: count, dtype: int64



TOP VALUES FOR: INSTITUTION_NAM

# Zenodo

In [71]:
Zenodo_df, Zenodo_summary = get_server_data("Zenodo")


 SERVER ANALYSIS: ZENODO
  > Files found:    1
  > Raw records:    166786
  > Cleaned shape:  (166786, 91)
  > Unique DOIs:    166786
--------------------------------------------------

TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
------------------------------
doi_prefix_first_token
10.5281/zenodo    166786
Name: count, dtype: int64



TOP VALUES FOR: PRIMARY_DOMAIN
------------------------------
primary_domain
zenodo.org    166786
Name: count, dtype: int64



TOP VALUES FOR: PREFIX
------------------------------
prefix
10.5281    166786
Name: count, dtype: int64



TOP VALUES FOR: MEMBER_ID
------------------------------
member_id
None    166786
Name: count, dtype: int64



TOP VALUES FOR: PUBLISHER
------------------------------
publisher
Zenodo                                                                            163441
The Five Principles of Organized Complexity.                                         385
Landon Puritz                                                              